In [ ]:
# old corrected augs and two RandRotate based on teses

import torch.nn.functional as F
import os, tarfile, zipfile, json, time
from typing import Optional, Tuple

def _human(n):
    for u in ["B","KB","MB","GB","TB"]:
        if n < 1024: return f"{n:.1f} {u}"
        n /= 1024
    return f"{n:.1f} PB"

def prepare_dataset_from_archive(
    cfg,
    archive_path: str,
    extract_dir: str,
    ensure_cache_dirs: Optional[Tuple[str, str]] = None,
    force_reextract: bool = False,
) -> str:
    """ نسخه آپدیت شده: پشتیبانی کامل از TAR و ZIP """
    if not os.path.exists(archive_path):
        raise FileNotFoundError(f"Archive not found: {archive_path}")

    os.makedirs(extract_dir, exist_ok=True)

    # اگر پوشه پر است و فورس نکردیم، همان را برگردان
    if not force_reextract and len(os.listdir(extract_dir)) > 0:
        print(f"[dataset] Found existing data in {extract_dir}")
        # پیدا کردن پوشه داخلی
        subs = [d for d in os.listdir(extract_dir) if os.path.isdir(os.path.join(extract_dir, d))]
        return os.path.join(extract_dir, subs[0]) if len(subs) == 1 else extract_dir

    print(f"[dataset] extracting: {archive_path} -> {extract_dir}")

    if archive_path.endswith(".tar"):
        with tarfile.open(archive_path, "r") as tar:
            tar.extractall(path=extract_dir)
    elif archive_path.endswith(".zip"):
        with zipfile.ZipFile(archive_path, "r") as z:
            z.extractall(extract_dir)
    else:
        raise ValueError("Format not supported (only .tar or .zip)")

    print("Done.")

    # پیدا کردن پوشه ریشه دیتاست
    subs = [d for d in os.listdir(extract_dir) if os.path.isdir(os.path.join(extract_dir, d))]
    final_root = os.path.join(extract_dir, subs[0]) if len(subs) == 1 else extract_dir
    return final_root

# section 1
import os, glob
import numpy as np
import nibabel as nib

def _load_nii(path, to_float=True, canonical=True):
    img = nib.load(path)
    if canonical:
        img = nib.as_closest_canonical(img)  # RAS
    arr = img.get_fdata(dtype=np.float32 if to_float else None)
    return arr

def _find_one(folder, patterns):
    """
    اولین فایلی که با یکی از الگوها match شود را برمی‌گرداند.
    الگوها باید شامل wildcard باشند (مثلاً *.nii* تا .nii و .nii.gz را بگیرد).
    """
    for p in patterns:
        m = sorted(glob.glob(os.path.join(folder, p)))
        if m:
            return m[0]
    raise FileNotFoundError(f"Not found in {folder}: {patterns}")

def load_modalities_from_folder(folder_name: str, image_id: str = ""):

    """
    خروجی:
      image: (4, W, H,  D) ترتیب [T1, T1Gd/T1ce, T2, FLAIR] (float32)
      mask : ( W, H, D) (int64)
    - به‌صورت محافظه‌کارانه، الگوها چندین نام رایج را پوشش می‌دهند.
    - اگر image_id ندادی، باز هم با الگوهای عمومی کار می‌کند.
    """
    # ---- الگوهای قابل‌قبول برای هر مودالیتی ----
    # توجه: ترتیب الگوها از خاص به عام است تا اشتباه match نشود.
    t1_patterns = [
        f"*{image_id}*t1.nii*", "*_t1.nii*", "*t1w.nii*", "*t1w*.nii*", "*t1n*.nii*"
    ]
    t1ce_patterns = [
        f"*{image_id}*t1ce.nii*", "*t1ce.nii*", "*t1c.nii*", "*t1gd.nii*", "*t1Gd.nii*"
    ]
    t2_patterns = [
        f"*{image_id}*t2.nii*", "*_t2.nii*", "*t2w.nii*"
    ]
    flair_patterns = [
        f"*{image_id}*flair.nii*", "*_flair.nii*", "*t2flair.nii*", "*t2f.nii*"
    ]
    seg_patterns = [
        f"*{image_id}*seg.nii*", "*_seg.nii*", "*segmentation*.nii*"
    ]

    # ---- پیدا کردن مسیر فایل‌ها ----
    t1n = _find_one(folder_name, t1_patterns)
    t1c = _find_one(folder_name, t1ce_patterns)
    t2w = _find_one(folder_name, t2_patterns)
    t2f = _find_one(folder_name, flair_patterns)
    seg = _find_one(folder_name, seg_patterns)

    # ---- لود داده‌ها ----
    a_t1  = _load_nii(t1n,  to_float=True, canonical=True)   # (w,h,d)
    a_t1c = _load_nii(t1c,  to_float=True, canonical=True)
    a_t2  = _load_nii(t2w,  to_float=True, canonical=True)
    a_fl  = _load_nii(t2f,  to_float=True, canonical=True)
    m_seg = _load_nii(seg,  to_float=False, canonical=True ).astype(np.int64)  # (W,H,D)


    # ---- sanity check شکل‌ها (فقط اخطار) ----
    shapes = {tuple(a_t1.shape), tuple(a_t1c.shape), tuple(a_t2.shape), tuple(a_fl.shape), tuple(m_seg.shape)}
    if len(shapes) != 1:
        print(f"[warn] mismatched shapes in {folder_name}: {shapes}")

    # ---- استک کانال-اول ----
    image = np.stack([a_t1, a_t1c, a_t2, a_fl], axis=0).astype(np.float32)  # (4,W,H,D)
    mask  = m_seg  # (W, H, D)  int64

    #print('         ------   loaded_modalities_from_folder ✅  ---------    ')

    return image, mask
# section 2
import numpy as np

def remap_label_4to3(mask: np.ndarray) -> np.ndarray:
    """
    BraTS: کلاس 4 → 3 (ET)
    ورودی: (H,W,D) عددی
    خروجی: (H,W,D) int64
    """
    out = mask.astype(np.int64, copy=True)
    out[out == 4] = 3
    return out

def center_pad_crop(arr: np.ndarray, out_shape, pad_val=0.0):
    """
    آرایه را به out_shape به‌صورت مرکز-محور پد/کراپ می‌کند.
    با ترتیب (W, H, D) سازگار است.
    """
    # تشخیص تعداد کانال‌ها
    if arr.ndim == 4:
        # ورودی: (C, W, H, D)
        C, W, H, D = arr.shape
        tgt_W, tgt_H, tgt_D = out_shape
        out = np.full((C, tgt_W, tgt_H, tgt_D), pad_val, dtype=arr.dtype)
    elif arr.ndim == 3:
        # ورودی: (W, H, D)
        W, H, D = arr.shape
        tgt_W, tgt_H, tgt_D = out_shape
        out = np.full((tgt_W, tgt_H, tgt_D), pad_val, dtype=arr.dtype)
    else:
        raise ValueError("آرایه باید 3 یا 4 بعدی باشد.")

    def get_sl(src_len, dst_len):
        if src_len >= dst_len:
            # Crop
            start = (src_len - dst_len) // 2
            return slice(start, start + dst_len), slice(0, dst_len)
        else:
            # Pad
            start = (dst_len - src_len) // 2
            return slice(0, src_len), slice(start, start + src_len)

    # محاسبه اسلایس‌ها برای هر بعد
    sW, dW = get_sl(W, tgt_W)
    sH, dH = get_sl(H, tgt_H)
    sD, dD = get_sl(D, tgt_D)

    if arr.ndim == 4:
        out[:, dW, dH, dD] = arr[:, sW, sH, sD]
    else:
        out[dW, dH, dD] = arr[sW, sH, sD]

    return out

def make_brain_mask(image: np.ndarray, pad_val=0.0) -> np.ndarray:
    """
    mask باینری مغز از روی خود تصویر:
      - هر وکسل که همه‌ی کانال‌هاش == pad_val باشد، خارج مغز است.
      - بقیه مغز.
    ورودی: image شکل (C, W, H, D)
    خروجی: (W, H, D) از نوع bool

    """
    assert image.ndim == 4
    return np.any(image != pad_val, axis=0)


# =========================================================
# Section 3: Dataset Class (BraTS Dict Dataset) — FIXED 💎
# =========================================================
import os, random
import numpy as np
import torch
from torch.utils.data import Dataset
from monai.data import MetaTensor

def list_case_folders(dataset_path):
    return [d for d in sorted(os.listdir(dataset_path))
            if os.path.isdir(os.path.join(dataset_path, d))]

'''
def list_case_folders(dataset_path):
    """
    لیست فولدرها را می‌گیرد و اگر تعداد محدود شده باشد، فقط همان مقدار را برمی‌گرداند.
    """
    max_samples=500
    # گرفتن تمام فولدرهای معتبر
    all_cases = [d for d in sorted(os.listdir(dataset_path))
                 if os.path.isdir(os.path.join(dataset_path, d))]

    # اگر کاربر عددی را در CFG ست کرده باشد
    if max_samples is not None and max_samples > 0:
        print(f"⚠️ Limiting dataset to first {max_samples} cases (out of {len(all_cases)})")
        return all_cases[:max_samples]

    return all_cases

'''


def create_case_splits(dataset_path, kfold=5, fold=0, seed=42):
    cases_all = list_case_folders(dataset_path)
    rng = random.Random(seed)
    rng.shuffle(cases_all)
    idx = np.arange(len(cases_all))
    folds = np.array_split(idx, kfold)
    val_idx = set(folds[fold])
    train_cases = [cases_all[i] for i in idx if i not in val_idx]
    val_cases   = [cases_all[i] for i in idx if i in val_idx]
    return train_cases, val_cases

class BraTSDictDataset(Dataset):
    def __init__(self, dataset_path, training=True, kfold=5, fold=0,
                 out_shape=None, remap_4to3=True, pad_val=0.0, seed=42, verbose=False):
        super().__init__()
        self.dataset_path = dataset_path
        self.training = training
        self.out_shape = out_shape
        self.remap_4to3 = remap_4to3
        self.pad_val = pad_val
        self.verbose = verbose

        train_cases, val_cases = create_case_splits(dataset_path, kfold=kfold, fold=fold, seed=seed)
        self.cases = train_cases if training else val_cases

        if verbose:
            split = "train" if training else "val"
            print(f"[{split}] cases={len(self.cases)}")

    def __len__(self):
        return len(self.cases)

    def __getitem__(self, idx):
        case_id = self.cases[idx]
        folder = os.path.join(self.dataset_path, case_id)

        # img: (4, W, H, D)
        # lbl: (W, H, D)
        img, lbl = load_modalities_from_folder(folder_name=folder, image_id=case_id)

        if self.remap_4to3:
            lbl = remap_label_4to3(lbl)

        if self.out_shape is not None:
            img = center_pad_crop(img, self.out_shape, pad_val=self.pad_val)
            lbl = center_pad_crop(lbl, self.out_shape, pad_val=0)

        brain_mask = make_brain_mask(img, pad_val=self.pad_val)

        # --- 🔴 اصلاح حیاتی: اضافه کردن بعد کانال به لیبل ---
        # لیبل از (W,H,D) به (1,W,H,D) تبدیل می‌شود تا Spacingd گیج نشود.
        lbl = lbl[None, ...]

        # تبدیل به MetaTensor برای حفظ اطلاعات ابعاد
        img_t = MetaTensor(torch.from_numpy(img.astype(np.float32)), affine=torch.eye(4))
        lbl_t = MetaTensor(torch.from_numpy(lbl.astype(np.float32)), affine=torch.eye(4)) # موقتا فلوت برای اینترپولیشن
        bm_t  = torch.from_numpy(brain_mask.astype(bool))

        return {"image": img_t, "label": lbl_t, "brain_mask": bm_t, "case_id": case_id}

# section 4

def mk_brain_mask_lambdad(cfg):
    tol = float(getattr(cfg, "bg_tol", 1e-6))
    pad_val = float(getattr(cfg, "pad_val", 0.0))

    def _f(im):
        # تبدیل به float برای محاسبات دقیق
        im32 = im.float()

        # ۱. تشخیص نقاطی که پدینگ یا صفر مطلق نیستند
        # نکته: استفاده از im32.abs() > 1e-4 برای اطمینان از حذف نویزهای میکروسکوپی
        m_pad = (im32 != pad_val) & (torch.abs(im32) > tol)

        # ۲. اگر در هر یک از ۴ مدالیته (T1, T2, ...) سیگنالی وجود داشته باشد، آنجا را مغز فرض کن
        m = m_pad.any(dim=0)

        # ۳. (اختیاری اما پیشنهادی) پر کردن سوراخ‌های احتمالی داخل ماسک
        # اگر در آینده متوجه شدی داخل تومورها "سیاه" می‌افتد، اینجا می‌توان از
        # عملیات مورفولوژیک استفاده کرد. اما برای شروع همین کافی است.

        return m.unsqueeze(0).float() # خروجی (1, W, H, D) به صورت 0 و 1

    return Lambdad(keys=["brain_mask_tmp"], func=_f)
# section 5

def dilate_label_bin_lambdad(kernel: int = 7):
    assert kernel % 2 == 1, "kernel باید فرد باشد."
    pad = kernel // 2

    def _f(x):
        # x شکلش هست (1, W, H, D)
        if x.ndim == 3:
            x = x.unsqueeze(0)

        # اطمینان از باینری بودن و نوع داده
        x = (x > 0).float()

        # بردن به فرمت (N, C, D, H, W) که برای max_pool3d بهینه است
        # نکته: ترتیب W,H,D مهم نیست چون کرنل متقارن است، فقط باید 5 بعدی شود
        xb = x.unsqueeze(0) # (1, 1, W, H, D)

        # عملیات دیلیت (بزرگتر کردن ناحیه تومور)
        y = F.max_pool3d(xb, kernel_size=kernel, stride=1, padding=pad)

        # برگشت به شکل (1, W, H, D) و تبدیل به uint8 برای اشغال حافظه کمتر
        return y.squeeze(0).to(torch.uint8)

    return Lambdad(keys=["label_bin_tmp"], func=_f)
import torch
import torch.nn.functional as F
from monai.transforms import (
    Compose, EnsureTyped, Orientationd, Spacingd, Lambdad,
    CropForegroundd, CopyItemsd, MaskIntensityd,
    NormalizeIntensityd, SpatialPadd, DeleteItemsd,
    RandCropByPosNegLabeld, RandBiasFieldd, RandFlipd,
    RandRotate90d, RandGaussianNoised, RandGaussianSmoothd,
    RandScaleIntensityd, RandShiftIntensityd, RandAdjustContrastd
)

# =========================================================
# Section 6 & 7: Pre-Processing (Train & Val)
# =========================================================
'''def build_pre_transform_train(cfg):
    return Compose([
        EnsureTyped(keys=["image", "label"], track_meta=True),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        Spacingd(
            keys=["image", "label"],
            pixdim=(1.0, 1.0, 1.0),
            mode=("bilinear", "nearest")
        ),
        CropForegroundd(keys=["image", "label"], source_key="image"),

        # استراتژی نرمال‌سازی اصلی شما (تکرار ماسک برای دقت بالا)
        CopyItemsd(keys=["image"], names=["brain_mask_tmp"]),
        mk_brain_mask_lambdad(cfg),
        MaskIntensityd(keys=["image"], mask_key="brain_mask_tmp"),
        NormalizeIntensityd(keys=["image"], channel_wise=True, nonzero=True),
        MaskIntensityd(keys=["image"], mask_key="brain_mask_tmp"),

        SpatialPadd(keys=["image", "label", "brain_mask_tmp"], spatial_size=cfg.patch_size),
        CopyItemsd(keys=["label"], names=["label_bin_tmp"]),
        Lambdad(keys=["label_bin_tmp"], func=lambda x: (x > 0).float()),
        dilate_label_bin_lambdad(kernel=getattr(cfg, "pos_dilate_kernel", 7)),
    ])
    '''
def build_pre_transform_train(cfg):
    return Compose([
        EnsureTyped(keys=["image", "label"], track_meta=True),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        Spacingd(
            keys=["image", "label"],
            pixdim=(1.0, 1.0, 1.0),
            mode=("bilinear", "nearest")
        ),
        CropForegroundd(keys=["image", "label"], source_key="image"),

        CopyItemsd(keys=["image"], names=["brain_mask_tmp"]),
        mk_brain_mask_lambdad(cfg),
        MaskIntensityd(keys=["image"], mask_key="brain_mask_tmp"),
        NormalizeIntensityd(keys=["image"], channel_wise=True, nonzero=True),
        MaskIntensityd(keys=["image"], mask_key="brain_mask_tmp"),

        SpatialPadd(keys=["image", "label", "brain_mask_tmp"], spatial_size=cfg.patch_size),

        # --- تغییر مهم اینجاست ---
        # ایجاد یک ماسک که هم تومور (WT) و هم با اولویت بیشتر ET را شامل شود
        CopyItemsd(keys=["label"], names=["sampling_map"]),
        # کلاس 3 در کد شما همان ET است. وزن آن را در نمونه‌برداری بالا می‌بریم
        Lambdad(keys=["sampling_map"], func=lambda x: (x > 0).float() + (x == 3).float() * 2.0),
    ])

def build_pre_transform_val(cfg):
    return Compose([
        EnsureTyped(keys=["image", "label"], track_meta=True),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        Spacingd(
            keys=["image", "label"],
            pixdim=(1.0, 1.0, 1.0),
            mode=("bilinear", "nearest")
        ),
        Lambdad(keys=["label"], func=lambda x: x.unsqueeze(0) if x.ndim == 3 else x),
        CropForegroundd(keys=["image", "label"], source_key="image"),

        CopyItemsd(keys=["image"], names=["brain_mask_tmp"]),
        mk_brain_mask_lambdad(cfg),
        MaskIntensityd(keys=["image"], mask_key="brain_mask_tmp"),
        NormalizeIntensityd(keys=["image"], channel_wise=True, nonzero=True),
        MaskIntensityd(keys=["image"], mask_key="brain_mask_tmp"),

        SpatialPadd(keys=["image", "label", "brain_mask_tmp"], spatial_size=cfg.patch_size),
        Lambdad(keys=["label"], func=lambda x: x.long()),
        Lambdad(keys=["label"], func=lambda x: x.squeeze(0) if (x.ndim == 4 and x.shape[0] == 1) else x),
        #DeleteItemsd(keys=["brain_mask_tmp"]),
    ])

# =========================================================
# Section 9: Heavy Augmentation (The "Overfitting-Friendly" Version)
# =========================================================
'''def build_aug_transform(cfg):
    from monai.transforms import RandRotated

    return Compose([
        # 1. نمونه‌برداری هوشمند (Crop)
        RandCropByPosNegLabeld(
            keys=["image", "label", "brain_mask_tmp", "label_bin_tmp"],
            label_key="label_bin_tmp",
            spatial_size=cfg.patch_size,
            pos=2.0,  # تمرکز ۲ برابری روی نواحی دارای تومور
            neg=1.0,
            num_samples=getattr(cfg, "num_samples", 1),
            image_key="image",
            image_threshold=0,
        ),

        # 2. آگمنتیشن‌های هندسی (Geometric) - ترکیبی از کد تو و مقاله
        RandFlipd(keys=["image", "label", "brain_mask_tmp"], prob=0.5, spatial_axis=[0, 1, 2]),

        # چرخش ۹۰ درجه (ایمن و بدون درون‌یابی)
        RandRotate90d(keys=["image", "label", "brain_mask_tmp"], prob=0.5, max_k=3),

        # چرخش آزاد (طبق مقاله SOTA برای مقاومت در برابر زاویه سر)
        RandRotated(
            keys=["image", "label", "brain_mask_tmp"],
            prob=0.3,
            range_x=0.2, range_y=0.2, range_z=0.2, # حدود ۱۱ درجه انحراف
            mode=("bilinear", "nearest", "nearest"), # لیبل‌ها هرگز نباید bilinear شوند
            padding_mode="zeros"
        ),

        # 3. آگمنتیشن‌های شدت (Intensity) - بخش قدرتمند کد خودت
        RandBiasFieldd(keys=["image"], prob=0.3, coeff_range=(0.2, 0.3)),
        RandGaussianNoised(keys=["image"], prob=0.15, mean=0.0, std=0.01),
        RandGaussianSmoothd(keys=["image"], prob=0.2, sigma_x=(0.5, 1.0), sigma_y=(0.5, 1.0), sigma_z=(0.5, 1.0)),

        RandScaleIntensityd(keys=["image"], factors=0.10, prob=0.5),
        RandShiftIntensityd(keys=["image"], offsets=0.10, prob=0.5),

        # گامای قدرتمند برای یادگیری بافت (Texture)
        RandAdjustContrastd(keys=["image"], gamma=(0.7, 1.5), prob=0.5),

        # 4. پس‌پردازش و پاکسازی
        MaskIntensityd(keys=["image"], mask_key="brain_mask_tmp"),

        Lambdad(keys=["label"], func=lambda x: x.long()),
        # اطمینان از حذف بعد اضافه اگر وجود داشت
        Lambdad(keys=["label"], func=lambda x: x.squeeze(0) if (x.ndim == 4 and x.shape[0] == 1) else x),

        # حذف آیتم‌های کمکی برای بهینه‌سازی حافظه RAM
        DeleteItemsd(keys=["brain_mask_tmp", "label_bin_tmp"]),
        EnsureTyped(keys=["image", "label"]),
    ])'''

from monai.transforms import RandRotated
def build_aug_transform(cfg):


    return Compose([
        # 1. نمونه‌برداری هوشمند با تمرکز بر ET
        RandCropByPosNegLabeld(
            keys=["image", "label", "brain_mask_tmp"],
            label_key="sampling_map", # از نقشه جدیدی که ساختیم استفاده کن
            spatial_size=cfg.patch_size,
            pos=3.0,  # افزایش تمرکز بر نواحی مثبت (مخصوصا ET)
            neg=1.0,
            num_samples=getattr(cfg, "num_samples", 1),
            image_key="image",
            image_threshold=0,
        ),

        RandFlipd(keys=["image", "label", "brain_mask_tmp"], prob=0.5, spatial_axis=[0, 1, 2]),
        RandRotate90d(keys=["image", "label", "brain_mask_tmp"], prob=0.5, max_k=3),

        # 2. اصلاح گاما (بازه را محدود کردیم تا کنتراست T1ce حفظ شود)
        RandAdjustContrastd(keys=["image"], gamma=(0.7, 1.5), prob=0.3), # بازه معقول‌تر

        # 3. اصلاح چرخش آزاد (کاهش احتمال برای جلوگیری از مات شدن لبه‌های ET)
        RandRotated(
            keys=["image", "label", "brain_mask_tmp"],
            prob=0.2, # کاهش احتمال از 0.3 به 0.2
            range_x=0.15, range_y=0.15, range_z=0.15,
            mode=("bilinear", "nearest", "nearest"),
            padding_mode="zeros"
        ),

        RandBiasFieldd(keys=["image"], prob=0.2, coeff_range=(0.1, 0.2)), # ملایم‌تر کردن نویز
        RandGaussianNoised(keys=["image"], prob=0.15, mean=0.0, std=0.01),

        # 4. پاکسازی نهایی
        MaskIntensityd(keys=["image"], mask_key="brain_mask_tmp"),

        Lambdad(keys=["label"], func=lambda x: x.long()),
        Lambdad(keys=["label"], func=lambda x: x.squeeze(0) if (x.ndim == 4 and x.shape[0] == 1) else x),

        # حذف آیتم‌های اضافی
        DeleteItemsd(keys=["brain_mask_tmp", "sampling_map"]),
        EnsureTyped(keys=["image", "label"]),
    ])
# =========================================================
# Section 10: Data Builders (Caching & Loading Strategy) ✅
# =========================================================
import random
import numpy as np
import torch
from pathlib import Path
from monai.data import (
    Dataset as MonaiDataset, CacheDataset, PersistentDataset,
    DataLoader, list_data_collate
)

# ---------------------------------------------------------
# Helper Functions for Reproducibility
# ---------------------------------------------------------
def _seed_worker(worker_id: int):
    """تنظیم سید برای هر ورکر جهت اطمینان از تکرارپذیری"""
    worker_seed = torch.initial_seed() % 2**32
    random.seed(worker_seed)
    np.random.seed(worker_seed)

def _make_generator(seed: int | None):
    """ساخت جنریتور رندوم برای دیتالودر"""
    if seed is None: return None
    g = torch.Generator()
    g.manual_seed(int(seed))
    return g

# ---------------------------------------------------------
# 1. Dataset Builder (Two-Stage Pipeline)
# ---------------------------------------------------------
def build_train_datasets(base_ds, cfg, pre_transform, aug_transform):
    """
    ساخت دیتاست آموزشی با استراتژی کشینگ (Cache Strategy).
    """
    backend = str(getattr(cfg, "cache_backend", "none")).lower()
    cache_rate = float(getattr(cfg, "cache_rate", 0.0))
    nw = int(getattr(cfg, "num_workers", 2))

    # --- مرحله 1: کش کردن دیتای اولیه ---
    # اگر کش ریت 0 باشد، حتی اگر رم انتخاب شده باشد، باید کش غیرفعال شود
    if backend == "ram" and cache_rate > 0:
        print(f"🚀 [Cache] Loading {cache_rate*100}% data into RAM with {nw} workers...")
        pre_ds = CacheDataset(
            data=base_ds,
            transform=pre_transform,
            cache_rate=cache_rate,
            num_workers=nw,
        )
    elif backend == "disk":
        cache_dir = Path(getattr(cfg, "cache_dir", "./cache_pre"))
        cache_dir.mkdir(parents=True, exist_ok=True)
        print(f"💾 [Cache] Using Disk Cache at: {cache_dir}")
        pre_ds = PersistentDataset(
            data=base_ds,
            transform=pre_transform,
            cache_dir=str(cache_dir),
        )
    else:
        # حالت None یا وقتی که cache_rate=0 است
        if backend == "ram": print("⚠️ Cache Rate is 0.0 -> Switching to standard Dataset.")
        pre_ds = MonaiDataset(data=base_ds, transform=pre_transform)

    # --- مرحله 2: اعمال آگمنتیشن‌های تصادفی روی دیتای کش شده ---
    train_ds = MonaiDataset(data=pre_ds, transform=aug_transform)
    return train_ds

# ---------------------------------------------------------
# 2. Train Loader Builder
# ---------------------------------------------------------
def build_train_loader(train_ds, cfg):
    """
    ساخت دیتالودر مخصوص آموزش با list_data_collate
    """
    bs = int(getattr(cfg, "batch_size", 1))
    nw = int(getattr(cfg, "num_workers", 2))
    pin = bool(getattr(cfg, "pin_memory", True))
    seed = getattr(cfg, "seed", None)

    # اگر تعداد ورکرها 0 باشد، persistent_workers باید خاموش باشد
    persistent = (nw > 0)

    return DataLoader(
        dataset=train_ds,
        batch_size=bs,
        shuffle=True,
        num_workers=nw,
        pin_memory=pin,
        drop_last=False,
        # حیاتی: چون خروجی RandCrop یک لیست است، باید با این تابع جمع شود
        collate_fn=list_data_collate,
        persistent_workers=persistent,
        worker_init_fn=_seed_worker,
        generator=_make_generator(seed),
        prefetch_factor=2 if nw > 0 else None
    )

# ---------------------------------------------------------
# 3. Validation Loader Builder
# ---------------------------------------------------------
def build_val_loader(val_base, cfg, pre_val_transform):
    """
    ساخت دیتالودر ولیدیشن (بدون شافل، بچ سایز 1)
    """
    # ولیدیشن آگمنتیشن ندارد، فقط pre_transform
    val_ds = MonaiDataset(data=val_base, transform=pre_val_transform)

    nw = int(getattr(cfg, "num_workers", 2))
    pin = bool(getattr(cfg, "pin_memory", True))
    seed = getattr(cfg, "seed", None)
    persistent = (nw > 0)

    return DataLoader(
        dataset=val_ds,
        batch_size=1, # ولیدیشن همیشه تک‌تصویر است
        shuffle=False,
        num_workers=nw,
        pin_memory=pin,
        persistent_workers=persistent,
        worker_init_fn=_seed_worker,
        generator=_make_generator(seed),
        prefetch_factor=2 if nw > 0 else None
    )

# =========================================================
# Section 11: Loader Builder — CLEAN VERSION ✅
# =========================================================

def build_loaders(cfg, data_root=None):

    if data_root is None:
        data_root = cfg.dataset_path

    print(f"📂 Loading Dataset from: {data_root}")

    # 1. ساخت دیتاست‌های پایه (Passing explicit configs)
    # حتماً kfold و seed را از cfg بگیرید تا تقسیم‌بندی درست باشد
    train_base = BraTSDictDataset(
        dataset_path=data_root,
        training=True,
        kfold=getattr(cfg, "kfold", 5),
        fold=getattr(cfg, "fold", 0),
        seed=getattr(cfg, "seed", 42),
        verbose=True
    )

    val_base = BraTSDictDataset(
        dataset_path=data_root,
        training=False,
        kfold=getattr(cfg, "kfold", 5),
        fold=getattr(cfg, "fold", 0),
        seed=getattr(cfg, "seed", 42),
        verbose=True
    )

    print(f"📊 [Stats] Train: {len(train_base)} | Val: {len(val_base)}")

    # 2. ساخت ترنسفرم‌ها
    print("⚙️ Building Transforms...")
    pre_tr  = build_pre_transform_train(cfg)
    aug_tr  = build_aug_transform(cfg)
    pre_val = build_pre_transform_val(cfg)

    # 3. ترکیب و ساخت لودرها
    print("🚀 Initializing Pipeline...")

    # ساخت دیتاست نهایی (شامل کشینگ در صورت نیاز)
    train_ds = build_train_datasets(train_base, cfg, pre_tr, aug_tr)

    # ساخت دیتالودرها
    train_loader = build_train_loader(train_ds, cfg)
    val_loader = build_val_loader(val_base, cfg, pre_val)

    print("✅ All Loaders Ready!")
    return train_loader, val_loader



In [ ]:
'''

cfg = CFG()


#ZIP_PATH   = "/content/data/BraTS/BraTS2021_Training_Data.tar"

ZIP_PATH   = "/content/drive/My Drive/brats2021_120.zip"

EXTRACT_TO = "/content/data/BraTS"      # این‌جا بدون تودرتو می‌ریزه
train_cache  = "/content/cache_pre_train"
val_cache    = "/content/cache_pre_val"

dataset_root = prepare_dataset_from_archive(
    cfg,
    archive_path=ZIP_PATH,
    extract_dir=EXTRACT_TO,
    ensure_cache_dirs=(train_cache, val_cache),
    force_reextract=True,
    )



cfg.dataset_path = dataset_root
print(cfg.dataset_path)


'''

In [ ]:
# old corrected augs and two RandRotate based on teses
# best sofar
'''
import torch.nn.functional as F
import os, tarfile, zipfile, json, time
from typing import Optional, Tuple

def _human(n):
    for u in ["B","KB","MB","GB","TB"]:
        if n < 1024: return f"{n:.1f} {u}"
        n /= 1024
    return f"{n:.1f} PB"

def prepare_dataset_from_archive(
    cfg,
    archive_path: str,
    extract_dir: str,
    ensure_cache_dirs: Optional[Tuple[str, str]] = None,
    force_reextract: bool = False,
) -> str:
    """ نسخه آپدیت شده: پشتیبانی کامل از TAR و ZIP """
    if not os.path.exists(archive_path):
        raise FileNotFoundError(f"Archive not found: {archive_path}")

    os.makedirs(extract_dir, exist_ok=True)

    # اگر پوشه پر است و فورس نکردیم، همان را برگردان
    if not force_reextract and len(os.listdir(extract_dir)) > 0:
        print(f"[dataset] Found existing data in {extract_dir}")
        # پیدا کردن پوشه داخلی
        subs = [d for d in os.listdir(extract_dir) if os.path.isdir(os.path.join(extract_dir, d))]
        return os.path.join(extract_dir, subs[0]) if len(subs) == 1 else extract_dir

    print(f"[dataset] extracting: {archive_path} -> {extract_dir}")

    if archive_path.endswith(".tar"):
        with tarfile.open(archive_path, "r") as tar:
            tar.extractall(path=extract_dir)
    elif archive_path.endswith(".zip"):
        with zipfile.ZipFile(archive_path, "r") as z:
            z.extractall(extract_dir)
    else:
        raise ValueError("Format not supported (only .tar or .zip)")

    print("Done.")

    # پیدا کردن پوشه ریشه دیتاست
    subs = [d for d in os.listdir(extract_dir) if os.path.isdir(os.path.join(extract_dir, d))]
    final_root = os.path.join(extract_dir, subs[0]) if len(subs) == 1 else extract_dir
    return final_root

# section 1
import os, glob
import numpy as np
import nibabel as nib

def _load_nii(path, to_float=True, canonical=True):
    img = nib.load(path)
    if canonical:
        img = nib.as_closest_canonical(img)  # RAS
    arr = img.get_fdata(dtype=np.float32 if to_float else None)
    return arr

def _find_one(folder, patterns):
    """
    اولین فایلی که با یکی از الگوها match شود را برمی‌گرداند.
    الگوها باید شامل wildcard باشند (مثلاً *.nii* تا .nii و .nii.gz را بگیرد).
    """
    for p in patterns:
        m = sorted(glob.glob(os.path.join(folder, p)))
        if m:
            return m[0]
    raise FileNotFoundError(f"Not found in {folder}: {patterns}")

def load_modalities_from_folder(folder_name: str, image_id: str = ""):

    """
    خروجی:
      image: (4, W, H,  D) ترتیب [T1, T1Gd/T1ce, T2, FLAIR] (float32)
      mask : ( W, H, D) (int64)
    - به‌صورت محافظه‌کارانه، الگوها چندین نام رایج را پوشش می‌دهند.
    - اگر image_id ندادی، باز هم با الگوهای عمومی کار می‌کند.
    """
    # ---- الگوهای قابل‌قبول برای هر مودالیتی ----
    # توجه: ترتیب الگوها از خاص به عام است تا اشتباه match نشود.
    t1_patterns = [
        f"*{image_id}*t1.nii*", "*_t1.nii*", "*t1w.nii*", "*t1w*.nii*", "*t1n*.nii*"
    ]
    t1ce_patterns = [
        f"*{image_id}*t1ce.nii*", "*t1ce.nii*", "*t1c.nii*", "*t1gd.nii*", "*t1Gd.nii*"
    ]
    t2_patterns = [
        f"*{image_id}*t2.nii*", "*_t2.nii*", "*t2w.nii*"
    ]
    flair_patterns = [
        f"*{image_id}*flair.nii*", "*_flair.nii*", "*t2flair.nii*", "*t2f.nii*"
    ]
    seg_patterns = [
        f"*{image_id}*seg.nii*", "*_seg.nii*", "*segmentation*.nii*"
    ]

    # ---- پیدا کردن مسیر فایل‌ها ----
    t1n = _find_one(folder_name, t1_patterns)
    t1c = _find_one(folder_name, t1ce_patterns)
    t2w = _find_one(folder_name, t2_patterns)
    t2f = _find_one(folder_name, flair_patterns)
    seg = _find_one(folder_name, seg_patterns)

    # ---- لود داده‌ها ----
    a_t1  = _load_nii(t1n,  to_float=True, canonical=True)   # (w,h,d)
    a_t1c = _load_nii(t1c,  to_float=True, canonical=True)
    a_t2  = _load_nii(t2w,  to_float=True, canonical=True)
    a_fl  = _load_nii(t2f,  to_float=True, canonical=True)
    m_seg = _load_nii(seg,  to_float=False, canonical=True ).astype(np.int64)  # (W,H,D)


    # ---- sanity check شکل‌ها (فقط اخطار) ----
    shapes = {tuple(a_t1.shape), tuple(a_t1c.shape), tuple(a_t2.shape), tuple(a_fl.shape), tuple(m_seg.shape)}
    if len(shapes) != 1:
        print(f"[warn] mismatched shapes in {folder_name}: {shapes}")

    # ---- استک کانال-اول ----
    image = np.stack([a_t1, a_t1c, a_t2, a_fl], axis=0).astype(np.float32)  # (4,W,H,D)
    mask  = m_seg  # (W, H, D)  int64

    #print('         ------   loaded_modalities_from_folder ✅  ---------    ')

    return image, mask
# section 2
import numpy as np

def remap_label_4to3(mask: np.ndarray) -> np.ndarray:
    """
    BraTS: کلاس 4 → 3 (ET)
    ورودی: (H,W,D) عددی
    خروجی: (H,W,D) int64
    """
    out = mask.astype(np.int64, copy=True)
    out[out == 4] = 3
    return out

def center_pad_crop(arr: np.ndarray, out_shape, pad_val=0.0):
    """
    آرایه را به out_shape به‌صورت مرکز-محور پد/کراپ می‌کند.
    با ترتیب (W, H, D) سازگار است.
    """
    # تشخیص تعداد کانال‌ها
    if arr.ndim == 4:
        # ورودی: (C, W, H, D)
        C, W, H, D = arr.shape
        tgt_W, tgt_H, tgt_D = out_shape
        out = np.full((C, tgt_W, tgt_H, tgt_D), pad_val, dtype=arr.dtype)
    elif arr.ndim == 3:
        # ورودی: (W, H, D)
        W, H, D = arr.shape
        tgt_W, tgt_H, tgt_D = out_shape
        out = np.full((tgt_W, tgt_H, tgt_D), pad_val, dtype=arr.dtype)
    else:
        raise ValueError("آرایه باید 3 یا 4 بعدی باشد.")

    def get_sl(src_len, dst_len):
        if src_len >= dst_len:
            # Crop
            start = (src_len - dst_len) // 2
            return slice(start, start + dst_len), slice(0, dst_len)
        else:
            # Pad
            start = (dst_len - src_len) // 2
            return slice(0, src_len), slice(start, start + src_len)

    # محاسبه اسلایس‌ها برای هر بعد
    sW, dW = get_sl(W, tgt_W)
    sH, dH = get_sl(H, tgt_H)
    sD, dD = get_sl(D, tgt_D)

    if arr.ndim == 4:
        out[:, dW, dH, dD] = arr[:, sW, sH, sD]
    else:
        out[dW, dH, dD] = arr[sW, sH, sD]

    return out

def make_brain_mask(image: np.ndarray, pad_val=0.0) -> np.ndarray:
    """
    mask باینری مغز از روی خود تصویر:
      - هر وکسل که همه‌ی کانال‌هاش == pad_val باشد، خارج مغز است.
      - بقیه مغز.
    ورودی: image شکل (C, W, H, D)
    خروجی: (W, H, D) از نوع bool

    """
    assert image.ndim == 4
    return np.any(image != pad_val, axis=0)


# =========================================================
# Section 3: Dataset Class (BraTS Dict Dataset) — FIXED 💎
# =========================================================
import os, random
import numpy as np
import torch
from torch.utils.data import Dataset
from monai.data import MetaTensor

def list_case_folders(dataset_path):
    return [d for d in sorted(os.listdir(dataset_path))
            if os.path.isdir(os.path.join(dataset_path, d))]

def create_case_splits(dataset_path, kfold=5, fold=0, seed=42):
    cases_all = list_case_folders(dataset_path)
    rng = random.Random(seed)
    rng.shuffle(cases_all)
    idx = np.arange(len(cases_all))
    folds = np.array_split(idx, kfold)
    val_idx = set(folds[fold])
    train_cases = [cases_all[i] for i in idx if i not in val_idx]
    val_cases   = [cases_all[i] for i in idx if i in val_idx]
    return train_cases, val_cases

class BraTSDictDataset(Dataset):
    def __init__(self, dataset_path, training=True, kfold=5, fold=0,
                 out_shape=None, remap_4to3=True, pad_val=0.0, seed=42, verbose=False):
        super().__init__()
        self.dataset_path = dataset_path
        self.training = training
        self.out_shape = out_shape
        self.remap_4to3 = remap_4to3
        self.pad_val = pad_val
        self.verbose = verbose

        train_cases, val_cases = create_case_splits(dataset_path, kfold=kfold, fold=fold, seed=seed)
        self.cases = train_cases if training else val_cases

        if verbose:
            split = "train" if training else "val"
            print(f"[{split}] cases={len(self.cases)}")

    def __len__(self):
        return len(self.cases)

    def __getitem__(self, idx):
        case_id = self.cases[idx]
        folder = os.path.join(self.dataset_path, case_id)

        # img: (4, W, H, D)
        # lbl: (W, H, D)
        img, lbl = load_modalities_from_folder(folder_name=folder, image_id=case_id)

        if self.remap_4to3:
            lbl = remap_label_4to3(lbl)

        if self.out_shape is not None:
            img = center_pad_crop(img, self.out_shape, pad_val=self.pad_val)
            lbl = center_pad_crop(lbl, self.out_shape, pad_val=0)

        brain_mask = make_brain_mask(img, pad_val=self.pad_val)

        # --- 🔴 اصلاح حیاتی: اضافه کردن بعد کانال به لیبل ---
        # لیبل از (W,H,D) به (1,W,H,D) تبدیل می‌شود تا Spacingd گیج نشود.
        lbl = lbl[None, ...]

        # تبدیل به MetaTensor برای حفظ اطلاعات ابعاد
        img_t = MetaTensor(torch.from_numpy(img.astype(np.float32)), affine=torch.eye(4))
        lbl_t = MetaTensor(torch.from_numpy(lbl.astype(np.float32)), affine=torch.eye(4)) # موقتا فلوت برای اینترپولیشن
        bm_t  = torch.from_numpy(brain_mask.astype(bool))

        return {"image": img_t, "label": lbl_t, "brain_mask": bm_t, "case_id": case_id}

# section 4

def mk_brain_mask_lambdad(cfg):
    tol = float(getattr(cfg, "bg_tol", 1e-6))
    pad_val = float(getattr(cfg, "pad_val", 0.0))

    def _f(im):
        # تبدیل به float برای محاسبات دقیق
        im32 = im.float()

        # ۱. تشخیص نقاطی که پدینگ یا صفر مطلق نیستند
        # نکته: استفاده از im32.abs() > 1e-4 برای اطمینان از حذف نویزهای میکروسکوپی
        m_pad = (im32 != pad_val) & (torch.abs(im32) > tol)

        # ۲. اگر در هر یک از ۴ مدالیته (T1, T2, ...) سیگنالی وجود داشته باشد، آنجا را مغز فرض کن
        m = m_pad.any(dim=0)

        # ۳. (اختیاری اما پیشنهادی) پر کردن سوراخ‌های احتمالی داخل ماسک
        # اگر در آینده متوجه شدی داخل تومورها "سیاه" می‌افتد، اینجا می‌توان از
        # عملیات مورفولوژیک استفاده کرد. اما برای شروع همین کافی است.

        return m.unsqueeze(0).float() # خروجی (1, W, H, D) به صورت 0 و 1

    return Lambdad(keys=["brain_mask_tmp"], func=_f)
# section 5

def dilate_label_bin_lambdad(kernel: int = 7):
    assert kernel % 2 == 1, "kernel باید فرد باشد."
    pad = kernel // 2

    def _f(x):
        # x شکلش هست (1, W, H, D)
        if x.ndim == 3:
            x = x.unsqueeze(0)

        # اطمینان از باینری بودن و نوع داده
        x = (x > 0).float()

        # بردن به فرمت (N, C, D, H, W) که برای max_pool3d بهینه است
        # نکته: ترتیب W,H,D مهم نیست چون کرنل متقارن است، فقط باید 5 بعدی شود
        xb = x.unsqueeze(0) # (1, 1, W, H, D)

        # عملیات دیلیت (بزرگتر کردن ناحیه تومور)
        y = F.max_pool3d(xb, kernel_size=kernel, stride=1, padding=pad)

        # برگشت به شکل (1, W, H, D) و تبدیل به uint8 برای اشغال حافظه کمتر
        return y.squeeze(0).to(torch.uint8)

    return Lambdad(keys=["label_bin_tmp"], func=_f)
import torch
import torch.nn.functional as F
from monai.transforms import (
    Compose, EnsureTyped, Orientationd, Spacingd, Lambdad,
    CropForegroundd, CopyItemsd, MaskIntensityd,
    NormalizeIntensityd, SpatialPadd, DeleteItemsd,
    RandCropByPosNegLabeld, RandBiasFieldd, RandFlipd,
    RandRotate90d, RandGaussianNoised, RandGaussianSmoothd,
    RandScaleIntensityd, RandShiftIntensityd, RandAdjustContrastd
)

# =========================================================
# Section 6 & 7: Pre-Processing (Train & Val)
# =========================================================
def build_pre_transform_train(cfg):
    return Compose([
        EnsureTyped(keys=["image", "label"], track_meta=True),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        Spacingd(
            keys=["image", "label"],
            pixdim=(1.0, 1.0, 1.0),
            mode=("bilinear", "nearest")
        ),
        CropForegroundd(keys=["image", "label"], source_key="image"),

        # استراتژی نرمال‌سازی اصلی شما (تکرار ماسک برای دقت بالا)
        CopyItemsd(keys=["image"], names=["brain_mask_tmp"]),
        mk_brain_mask_lambdad(cfg),
        MaskIntensityd(keys=["image"], mask_key="brain_mask_tmp"),
        NormalizeIntensityd(keys=["image"], channel_wise=True, nonzero=True),
        MaskIntensityd(keys=["image"], mask_key="brain_mask_tmp"),

        SpatialPadd(keys=["image", "label", "brain_mask_tmp"], spatial_size=cfg.patch_size),
        CopyItemsd(keys=["label"], names=["label_bin_tmp"]),
        Lambdad(keys=["label_bin_tmp"], func=lambda x: (x > 0).float()),
        dilate_label_bin_lambdad(kernel=getattr(cfg, "pos_dilate_kernel", 7)),
    ])

def build_pre_transform_val(cfg):
    return Compose([
        EnsureTyped(keys=["image", "label"], track_meta=True),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        Spacingd(
            keys=["image", "label"],
            pixdim=(1.0, 1.0, 1.0),
            mode=("bilinear", "nearest")
        ),
        Lambdad(keys=["label"], func=lambda x: x.unsqueeze(0) if x.ndim == 3 else x),
        CropForegroundd(keys=["image", "label"], source_key="image"),

        CopyItemsd(keys=["image"], names=["brain_mask_tmp"]),
        mk_brain_mask_lambdad(cfg),
        MaskIntensityd(keys=["image"], mask_key="brain_mask_tmp"),
        NormalizeIntensityd(keys=["image"], channel_wise=True, nonzero=True),
        MaskIntensityd(keys=["image"], mask_key="brain_mask_tmp"),

        SpatialPadd(keys=["image", "label", "brain_mask_tmp"], spatial_size=cfg.patch_size),
        Lambdad(keys=["label"], func=lambda x: x.long()),
        Lambdad(keys=["label"], func=lambda x: x.squeeze(0) if (x.ndim == 4 and x.shape[0] == 1) else x),
        DeleteItemsd(keys=["brain_mask_tmp"]),
    ])

# =========================================================
# Section 9: Heavy Augmentation (The "Overfitting-Friendly" Version)
# =========================================================
def build_aug_transform(cfg):
    from monai.transforms import RandRotated

    return Compose([
        # 1. نمونه‌برداری هوشمند (Crop)
        RandCropByPosNegLabeld(
            keys=["image", "label", "brain_mask_tmp", "label_bin_tmp"],
            label_key="label_bin_tmp",
            spatial_size=cfg.patch_size,
            pos=2.0,  # تمرکز ۲ برابری روی نواحی دارای تومور
            neg=1.0,
            num_samples=getattr(cfg, "num_samples", 1),
            image_key="image",
            image_threshold=0,
        ),

        # 2. آگمنتیشن‌های هندسی (Geometric) - ترکیبی از کد تو و مقاله
        RandFlipd(keys=["image", "label", "brain_mask_tmp"], prob=0.5, spatial_axis=[0, 1, 2]),

        # چرخش ۹۰ درجه (ایمن و بدون درون‌یابی)
        RandRotate90d(keys=["image", "label", "brain_mask_tmp"], prob=0.5, max_k=3),

        # چرخش آزاد (طبق مقاله SOTA برای مقاومت در برابر زاویه سر)
        RandRotated(
            keys=["image", "label", "brain_mask_tmp"],
            prob=0.3,
            range_x=0.2, range_y=0.2, range_z=0.2, # حدود ۱۱ درجه انحراف
            mode=("bilinear", "nearest", "nearest"), # لیبل‌ها هرگز نباید bilinear شوند
            padding_mode="zeros"
        ),

        # 3. آگمنتیشن‌های شدت (Intensity) - بخش قدرتمند کد خودت
        RandBiasFieldd(keys=["image"], prob=0.3, coeff_range=(0.2, 0.3)),
        RandGaussianNoised(keys=["image"], prob=0.15, mean=0.0, std=0.01),
        RandGaussianSmoothd(keys=["image"], prob=0.2, sigma_x=(0.5, 1.0), sigma_y=(0.5, 1.0), sigma_z=(0.5, 1.0)),

        RandScaleIntensityd(keys=["image"], factors=0.10, prob=0.5),
        RandShiftIntensityd(keys=["image"], offsets=0.10, prob=0.5),

        # گامای قدرتمند برای یادگیری بافت (Texture)
        RandAdjustContrastd(keys=["image"], gamma=(0.5, 4.5), prob=0.5),

        # 4. پس‌پردازش و پاکسازی
        MaskIntensityd(keys=["image"], mask_key="brain_mask_tmp"),

        Lambdad(keys=["label"], func=lambda x: x.long()),
        # اطمینان از حذف بعد اضافه اگر وجود داشت
        Lambdad(keys=["label"], func=lambda x: x.squeeze(0) if (x.ndim == 4 and x.shape[0] == 1) else x),

        # حذف آیتم‌های کمکی برای بهینه‌سازی حافظه RAM
        DeleteItemsd(keys=["brain_mask_tmp", "label_bin_tmp"]),
        EnsureTyped(keys=["image", "label"]),
    ])
# =========================================================
# Section 10: Data Builders (Caching & Loading Strategy) ✅
# =========================================================
import random
import numpy as np
import torch
from pathlib import Path
from monai.data import (
    Dataset as MonaiDataset, CacheDataset, PersistentDataset,
    DataLoader, list_data_collate
)

# ---------------------------------------------------------
# Helper Functions for Reproducibility
# ---------------------------------------------------------
def _seed_worker(worker_id: int):
    """تنظیم سید برای هر ورکر جهت اطمینان از تکرارپذیری"""
    worker_seed = torch.initial_seed() % 2**32
    random.seed(worker_seed)
    np.random.seed(worker_seed)

def _make_generator(seed: int | None):
    """ساخت جنریتور رندوم برای دیتالودر"""
    if seed is None: return None
    g = torch.Generator()
    g.manual_seed(int(seed))
    return g

# ---------------------------------------------------------
# 1. Dataset Builder (Two-Stage Pipeline)
# ---------------------------------------------------------
def build_train_datasets(base_ds, cfg, pre_transform, aug_transform):
    """
    ساخت دیتاست آموزشی با استراتژی کشینگ (Cache Strategy).
    """
    backend = str(getattr(cfg, "cache_backend", "none")).lower()
    cache_rate = float(getattr(cfg, "cache_rate", 0.0))
    nw = int(getattr(cfg, "num_workers", 2))

    # --- مرحله 1: کش کردن دیتای اولیه ---
    # اگر کش ریت 0 باشد، حتی اگر رم انتخاب شده باشد، باید کش غیرفعال شود
    if backend == "ram" and cache_rate > 0:
        print(f"🚀 [Cache] Loading {cache_rate*100}% data into RAM with {nw} workers...")
        pre_ds = CacheDataset(
            data=base_ds,
            transform=pre_transform,
            cache_rate=cache_rate,
            num_workers=nw,
        )
    elif backend == "disk":
        cache_dir = Path(getattr(cfg, "cache_dir", "./cache_pre"))
        cache_dir.mkdir(parents=True, exist_ok=True)
        print(f"💾 [Cache] Using Disk Cache at: {cache_dir}")
        pre_ds = PersistentDataset(
            data=base_ds,
            transform=pre_transform,
            cache_dir=str(cache_dir),
        )
    else:
        # حالت None یا وقتی که cache_rate=0 است
        if backend == "ram": print("⚠️ Cache Rate is 0.0 -> Switching to standard Dataset.")
        pre_ds = MonaiDataset(data=base_ds, transform=pre_transform)

    # --- مرحله 2: اعمال آگمنتیشن‌های تصادفی روی دیتای کش شده ---
    train_ds = MonaiDataset(data=pre_ds, transform=aug_transform)
    return train_ds

# ---------------------------------------------------------
# 2. Train Loader Builder
# ---------------------------------------------------------
def build_train_loader(train_ds, cfg):
    """
    ساخت دیتالودر مخصوص آموزش با list_data_collate
    """
    bs = int(getattr(cfg, "batch_size", 1))
    nw = int(getattr(cfg, "num_workers", 2))
    pin = bool(getattr(cfg, "pin_memory", True))
    seed = getattr(cfg, "seed", None)

    # اگر تعداد ورکرها 0 باشد، persistent_workers باید خاموش باشد
    persistent = (nw > 0)

    return DataLoader(
        dataset=train_ds,
        batch_size=bs,
        shuffle=True,
        num_workers=nw,
        pin_memory=pin,
        drop_last=False,
        # حیاتی: چون خروجی RandCrop یک لیست است، باید با این تابع جمع شود
        collate_fn=list_data_collate,
        persistent_workers=persistent,
        worker_init_fn=_seed_worker,
        generator=_make_generator(seed),
        prefetch_factor=2 if nw > 0 else None
    )

# ---------------------------------------------------------
# 3. Validation Loader Builder
# ---------------------------------------------------------
def build_val_loader(val_base, cfg, pre_val_transform):
    """
    ساخت دیتالودر ولیدیشن (بدون شافل، بچ سایز 1)
    """
    # ولیدیشن آگمنتیشن ندارد، فقط pre_transform
    val_ds = MonaiDataset(data=val_base, transform=pre_val_transform)

    nw = int(getattr(cfg, "num_workers", 2))
    pin = bool(getattr(cfg, "pin_memory", True))
    seed = getattr(cfg, "seed", None)
    persistent = (nw > 0)

    return DataLoader(
        dataset=val_ds,
        batch_size=1, # ولیدیشن همیشه تک‌تصویر است
        shuffle=False,
        num_workers=nw,
        pin_memory=pin,
        persistent_workers=persistent,
        worker_init_fn=_seed_worker,
        generator=_make_generator(seed),
        prefetch_factor=2 if nw > 0 else None
    )

# =========================================================
# Section 11: Loader Builder — CLEAN VERSION ✅
# =========================================================

def build_loaders(cfg, data_root=None):

    if data_root is None:
        data_root = cfg.dataset_path

    print(f"📂 Loading Dataset from: {data_root}")

    # 1. ساخت دیتاست‌های پایه (Passing explicit configs)
    # حتماً kfold و seed را از cfg بگیرید تا تقسیم‌بندی درست باشد
    train_base = BraTSDictDataset(
        dataset_path=data_root,
        training=True,
        kfold=getattr(cfg, "kfold", 5),
        fold=getattr(cfg, "fold", 0),
        seed=getattr(cfg, "seed", 42),
        verbose=True
    )

    val_base = BraTSDictDataset(
        dataset_path=data_root,
        training=False,
        kfold=getattr(cfg, "kfold", 5),
        fold=getattr(cfg, "fold", 0),
        seed=getattr(cfg, "seed", 42),
        verbose=True
    )

    print(f"📊 [Stats] Train: {len(train_base)} | Val: {len(val_base)}")

    # 2. ساخت ترنسفرم‌ها
    print("⚙️ Building Transforms...")
    pre_tr  = build_pre_transform_train(cfg)
    aug_tr  = build_aug_transform(cfg)
    pre_val = build_pre_transform_val(cfg)

    # 3. ترکیب و ساخت لودرها
    print("🚀 Initializing Pipeline...")

    # ساخت دیتاست نهایی (شامل کشینگ در صورت نیاز)
    train_ds = build_train_datasets(train_base, cfg, pre_tr, aug_tr)

    # ساخت دیتالودرها
    train_loader = build_train_loader(train_ds, cfg)
    val_loader = build_val_loader(val_base, cfg, pre_val)

    print("✅ All Loaders Ready!")
    return train_loader, val_loader


cfg = CFG()


#ZIP_PATH   = "/content/data/BraTS/BraTS2021_Training_Data.tar"

ZIP_PATH   = "/content/drive/My Drive/brats2021_120.zip"

EXTRACT_TO = "/content/data/BraTS"      # این‌جا بدون تودرتو می‌ریزه
train_cache  = "/content/cache_pre_train"
val_cache    = "/content/cache_pre_val"

dataset_root = prepare_dataset_from_archive(
    cfg,
    archive_path=ZIP_PATH,
    extract_dir=EXTRACT_TO,
    ensure_cache_dirs=(train_cache, val_cache),
    force_reextract=True,
    )



cfg.dataset_path = dataset_root
print(cfg.dataset_path)


'''

In [ ]:
# old corrected augs
'''
import torch.nn.functional as F
import os, tarfile, zipfile, json, time
from typing import Optional, Tuple

def _human(n):
    for u in ["B","KB","MB","GB","TB"]:
        if n < 1024: return f"{n:.1f} {u}"
        n /= 1024
    return f"{n:.1f} PB"

def prepare_dataset_from_archive(
    cfg,
    archive_path: str,
    extract_dir: str,
    ensure_cache_dirs: Optional[Tuple[str, str]] = None,
    force_reextract: bool = False,
) -> str:
    """ نسخه آپدیت شده: پشتیبانی کامل از TAR و ZIP """
    if not os.path.exists(archive_path):
        raise FileNotFoundError(f"Archive not found: {archive_path}")

    os.makedirs(extract_dir, exist_ok=True)

    # اگر پوشه پر است و فورس نکردیم، همان را برگردان
    if not force_reextract and len(os.listdir(extract_dir)) > 0:
        print(f"[dataset] Found existing data in {extract_dir}")
        # پیدا کردن پوشه داخلی
        subs = [d for d in os.listdir(extract_dir) if os.path.isdir(os.path.join(extract_dir, d))]
        return os.path.join(extract_dir, subs[0]) if len(subs) == 1 else extract_dir

    print(f"[dataset] extracting: {archive_path} -> {extract_dir}")

    if archive_path.endswith(".tar"):
        with tarfile.open(archive_path, "r") as tar:
            tar.extractall(path=extract_dir)
    elif archive_path.endswith(".zip"):
        with zipfile.ZipFile(archive_path, "r") as z:
            z.extractall(extract_dir)
    else:
        raise ValueError("Format not supported (only .tar or .zip)")

    print("Done.")

    # پیدا کردن پوشه ریشه دیتاست
    subs = [d for d in os.listdir(extract_dir) if os.path.isdir(os.path.join(extract_dir, d))]
    final_root = os.path.join(extract_dir, subs[0]) if len(subs) == 1 else extract_dir
    return final_root
   # section 1
import os, glob
import numpy as np
import nibabel as nib

def _load_nii(path, to_float=True, canonical=True):
    img = nib.load(path)
    if canonical:
        img = nib.as_closest_canonical(img)  # RAS
    arr = img.get_fdata(dtype=np.float32 if to_float else None)
    return arr

def _find_one(folder, patterns):
    """
    اولین فایلی که با یکی از الگوها match شود را برمی‌گرداند.
    الگوها باید شامل wildcard باشند (مثلاً *.nii* تا .nii و .nii.gz را بگیرد).
    """
    for p in patterns:
        m = sorted(glob.glob(os.path.join(folder, p)))
        if m:
            return m[0]
    raise FileNotFoundError(f"Not found in {folder}: {patterns}")

def load_modalities_from_folder(folder_name: str, image_id: str = ""):

    """
    خروجی:
      image: (4, W, H,  D) ترتیب [T1, T1Gd/T1ce, T2, FLAIR] (float32)
      mask : ( W, H, D) (int64)
    - به‌صورت محافظه‌کارانه، الگوها چندین نام رایج را پوشش می‌دهند.
    - اگر image_id ندادی، باز هم با الگوهای عمومی کار می‌کند.
    """
    # ---- الگوهای قابل‌قبول برای هر مودالیتی ----
    # توجه: ترتیب الگوها از خاص به عام است تا اشتباه match نشود.
    t1_patterns = [
        f"*{image_id}*t1.nii*", "*_t1.nii*", "*t1w.nii*", "*t1w*.nii*", "*t1n*.nii*"
    ]
    t1ce_patterns = [
        f"*{image_id}*t1ce.nii*", "*t1ce.nii*", "*t1c.nii*", "*t1gd.nii*", "*t1Gd.nii*"
    ]
    t2_patterns = [
        f"*{image_id}*t2.nii*", "*_t2.nii*", "*t2w.nii*"
    ]
    flair_patterns = [
        f"*{image_id}*flair.nii*", "*_flair.nii*", "*t2flair.nii*", "*t2f.nii*"
    ]
    seg_patterns = [
        f"*{image_id}*seg.nii*", "*_seg.nii*", "*segmentation*.nii*"
    ]

    # ---- پیدا کردن مسیر فایل‌ها ----
    t1n = _find_one(folder_name, t1_patterns)
    t1c = _find_one(folder_name, t1ce_patterns)
    t2w = _find_one(folder_name, t2_patterns)
    t2f = _find_one(folder_name, flair_patterns)
    seg = _find_one(folder_name, seg_patterns)

    # ---- لود داده‌ها ----
    a_t1  = _load_nii(t1n,  to_float=True, canonical=True)   # (w,h,d)
    a_t1c = _load_nii(t1c,  to_float=True, canonical=True)
    a_t2  = _load_nii(t2w,  to_float=True, canonical=True)
    a_fl  = _load_nii(t2f,  to_float=True, canonical=True)
    m_seg = _load_nii(seg,  to_float=False, canonical=True ).astype(np.int64)  # (W,H,D)


    # ---- sanity check شکل‌ها (فقط اخطار) ----
    shapes = {tuple(a_t1.shape), tuple(a_t1c.shape), tuple(a_t2.shape), tuple(a_fl.shape), tuple(m_seg.shape)}
    if len(shapes) != 1:
        print(f"[warn] mismatched shapes in {folder_name}: {shapes}")

    # ---- استک کانال-اول ----
    image = np.stack([a_t1, a_t1c, a_t2, a_fl], axis=0).astype(np.float32)  # (4,W,H,D)
    mask  = m_seg  # (W, H, D)  int64

    #print('         ------   loaded_modalities_from_folder ✅  ---------    ')

    return image, mask
# section 2
import numpy as np

def remap_label_4to3(mask: np.ndarray) -> np.ndarray:
    """
    BraTS: کلاس 4 → 3 (ET)
    ورودی: (H,W,D) عددی
    خروجی: (H,W,D) int64
    """
    out = mask.astype(np.int64, copy=True)
    out[out == 4] = 3
    return out

def center_pad_crop(arr: np.ndarray, out_shape, pad_val=0.0):
    """
    آرایه را به out_shape به‌صورت مرکز-محور پد/کراپ می‌کند.
    با ترتیب (W, H, D) سازگار است.
    """
    # تشخیص تعداد کانال‌ها
    if arr.ndim == 4:
        # ورودی: (C, W, H, D)
        C, W, H, D = arr.shape
        tgt_W, tgt_H, tgt_D = out_shape
        out = np.full((C, tgt_W, tgt_H, tgt_D), pad_val, dtype=arr.dtype)
    elif arr.ndim == 3:
        # ورودی: (W, H, D)
        W, H, D = arr.shape
        tgt_W, tgt_H, tgt_D = out_shape
        out = np.full((tgt_W, tgt_H, tgt_D), pad_val, dtype=arr.dtype)
    else:
        raise ValueError("آرایه باید 3 یا 4 بعدی باشد.")

    def get_sl(src_len, dst_len):
        if src_len >= dst_len:
            # Crop
            start = (src_len - dst_len) // 2
            return slice(start, start + dst_len), slice(0, dst_len)
        else:
            # Pad
            start = (dst_len - src_len) // 2
            return slice(0, src_len), slice(start, start + src_len)

    # محاسبه اسلایس‌ها برای هر بعد
    sW, dW = get_sl(W, tgt_W)
    sH, dH = get_sl(H, tgt_H)
    sD, dD = get_sl(D, tgt_D)

    if arr.ndim == 4:
        out[:, dW, dH, dD] = arr[:, sW, sH, sD]
    else:
        out[dW, dH, dD] = arr[sW, sH, sD]

    return out

def make_brain_mask(image: np.ndarray, pad_val=0.0) -> np.ndarray:
    """
    mask باینری مغز از روی خود تصویر:
      - هر وکسل که همه‌ی کانال‌هاش == pad_val باشد، خارج مغز است.
      - بقیه مغز.
    ورودی: image شکل (C, W, H, D)
    خروجی: (W, H, D) از نوع bool

    """
    assert image.ndim == 4
    return np.any(image != pad_val, axis=0)


# =========================================================
# Section 3: Dataset Class (BraTS Dict Dataset) — FIXED 💎
# =========================================================
import os, random
import numpy as np
import torch
from torch.utils.data import Dataset
from monai.data import MetaTensor

def list_case_folders(dataset_path):
    return [d for d in sorted(os.listdir(dataset_path))
            if os.path.isdir(os.path.join(dataset_path, d))]

def create_case_splits(dataset_path, kfold=5, fold=0, seed=42):
    cases_all = list_case_folders(dataset_path)
    rng = random.Random(seed)
    rng.shuffle(cases_all)
    idx = np.arange(len(cases_all))
    folds = np.array_split(idx, kfold)
    val_idx = set(folds[fold])
    train_cases = [cases_all[i] for i in idx if i not in val_idx]
    val_cases   = [cases_all[i] for i in idx if i in val_idx]
    return train_cases, val_cases

class BraTSDictDataset(Dataset):
    def __init__(self, dataset_path, training=True, kfold=5, fold=0,
                 out_shape=None, remap_4to3=True, pad_val=0.0, seed=42, verbose=False):
        super().__init__()
        self.dataset_path = dataset_path
        self.training = training
        self.out_shape = out_shape
        self.remap_4to3 = remap_4to3
        self.pad_val = pad_val
        self.verbose = verbose

        train_cases, val_cases = create_case_splits(dataset_path, kfold=kfold, fold=fold, seed=seed)
        self.cases = train_cases if training else val_cases

        if verbose:
            split = "train" if training else "val"
            print(f"[{split}] cases={len(self.cases)}")

    def __len__(self):
        return len(self.cases)

    def __getitem__(self, idx):
        case_id = self.cases[idx]
        folder = os.path.join(self.dataset_path, case_id)

        # img: (4, W, H, D)
        # lbl: (W, H, D)
        img, lbl = load_modalities_from_folder(folder_name=folder, image_id=case_id)

        if self.remap_4to3:
            lbl = remap_label_4to3(lbl)

        if self.out_shape is not None:
            img = center_pad_crop(img, self.out_shape, pad_val=self.pad_val)
            lbl = center_pad_crop(lbl, self.out_shape, pad_val=0)

        brain_mask = make_brain_mask(img, pad_val=self.pad_val)

        # --- 🔴 اصلاح حیاتی: اضافه کردن بعد کانال به لیبل ---
        # لیبل از (W,H,D) به (1,W,H,D) تبدیل می‌شود تا Spacingd گیج نشود.
        lbl = lbl[None, ...]

        # تبدیل به MetaTensor برای حفظ اطلاعات ابعاد
        img_t = MetaTensor(torch.from_numpy(img.astype(np.float32)), affine=torch.eye(4))
        lbl_t = MetaTensor(torch.from_numpy(lbl.astype(np.float32)), affine=torch.eye(4)) # موقتا فلوت برای اینترپولیشن
        bm_t  = torch.from_numpy(brain_mask.astype(bool))

        return {"image": img_t, "label": lbl_t, "brain_mask": bm_t, "case_id": case_id}

# section 4

def mk_brain_mask_lambdad(cfg):
    tol = float(getattr(cfg, "bg_tol", 1e-6))
    pad_val = float(getattr(cfg, "pad_val", 0.0))

    def _f(im):
        # تبدیل به float برای محاسبات دقیق
        im32 = im.float()

        # ۱. تشخیص نقاطی که پدینگ یا صفر مطلق نیستند
        # نکته: استفاده از im32.abs() > 1e-4 برای اطمینان از حذف نویزهای میکروسکوپی
        m_pad = (im32 != pad_val) & (torch.abs(im32) > tol)

        # ۲. اگر در هر یک از ۴ مدالیته (T1, T2, ...) سیگنالی وجود داشته باشد، آنجا را مغز فرض کن
        m = m_pad.any(dim=0)

        # ۳. (اختیاری اما پیشنهادی) پر کردن سوراخ‌های احتمالی داخل ماسک
        # اگر در آینده متوجه شدی داخل تومورها "سیاه" می‌افتد، اینجا می‌توان از
        # عملیات مورفولوژیک استفاده کرد. اما برای شروع همین کافی است.

        return m.unsqueeze(0).float() # خروجی (1, W, H, D) به صورت 0 و 1

    return Lambdad(keys=["brain_mask_tmp"], func=_f)
# section 5

def dilate_label_bin_lambdad(kernel: int = 7):
    assert kernel % 2 == 1, "kernel باید فرد باشد."
    pad = kernel // 2

    def _f(x):
        # x شکلش هست (1, W, H, D)
        if x.ndim == 3:
            x = x.unsqueeze(0)

        # اطمینان از باینری بودن و نوع داده
        x = (x > 0).float()

        # بردن به فرمت (N, C, D, H, W) که برای max_pool3d بهینه است
        # نکته: ترتیب W,H,D مهم نیست چون کرنل متقارن است، فقط باید 5 بعدی شود
        xb = x.unsqueeze(0) # (1, 1, W, H, D)

        # عملیات دیلیت (بزرگتر کردن ناحیه تومور)
        y = F.max_pool3d(xb, kernel_size=kernel, stride=1, padding=pad)

        # برگشت به شکل (1, W, H, D) و تبدیل به uint8 برای اشغال حافظه کمتر
        return y.squeeze(0).to(torch.uint8)

    return Lambdad(keys=["label_bin_tmp"], func=_f)
import torch
import torch.nn.functional as F
from monai.transforms import (
    Compose, EnsureTyped, Orientationd, Spacingd, Lambdad,
    CropForegroundd, CopyItemsd, MaskIntensityd,
    NormalizeIntensityd, SpatialPadd, DeleteItemsd,
    RandCropByPosNegLabeld, RandBiasFieldd, RandFlipd,
    RandRotate90d, RandGaussianNoised, RandGaussianSmoothd,
    RandScaleIntensityd, RandShiftIntensityd, RandAdjustContrastd
)

# =========================================================
# Section 6 & 7: Pre-Processing (Train & Val)
# =========================================================
def build_pre_transform_train(cfg):
    return Compose([
        EnsureTyped(keys=["image", "label"], track_meta=True),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        Spacingd(
            keys=["image", "label"],
            pixdim=(1.0, 1.0, 1.0),
            mode=("bilinear", "nearest")
        ),
        CropForegroundd(keys=["image", "label"], source_key="image"),

        # استراتژی نرمال‌سازی اصلی شما (تکرار ماسک برای دقت بالا)
        CopyItemsd(keys=["image"], names=["brain_mask_tmp"]),
        mk_brain_mask_lambdad(cfg),
        MaskIntensityd(keys=["image"], mask_key="brain_mask_tmp"),
        NormalizeIntensityd(keys=["image"], channel_wise=True, nonzero=True),
        MaskIntensityd(keys=["image"], mask_key="brain_mask_tmp"),

        SpatialPadd(keys=["image", "label", "brain_mask_tmp"], spatial_size=cfg.patch_size),
        CopyItemsd(keys=["label"], names=["label_bin_tmp"]),
        Lambdad(keys=["label_bin_tmp"], func=lambda x: (x > 0).float()),
        dilate_label_bin_lambdad(kernel=getattr(cfg, "pos_dilate_kernel", 7)),
    ])

def build_pre_transform_val(cfg):
    return Compose([
        EnsureTyped(keys=["image", "label"], track_meta=True),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        Spacingd(
            keys=["image", "label"],
            pixdim=(1.0, 1.0, 1.0),
            mode=("bilinear", "nearest")
        ),
        Lambdad(keys=["label"], func=lambda x: x.unsqueeze(0) if x.ndim == 3 else x),
        CropForegroundd(keys=["image", "label"], source_key="image"),

        CopyItemsd(keys=["image"], names=["brain_mask_tmp"]),
        mk_brain_mask_lambdad(cfg),
        MaskIntensityd(keys=["image"], mask_key="brain_mask_tmp"),
        NormalizeIntensityd(keys=["image"], channel_wise=True, nonzero=True),
        MaskIntensityd(keys=["image"], mask_key="brain_mask_tmp"),

        SpatialPadd(keys=["image", "label", "brain_mask_tmp"], spatial_size=cfg.patch_size),
        Lambdad(keys=["label"], func=lambda x: x.long()),
        Lambdad(keys=["label"], func=lambda x: x.squeeze(0) if (x.ndim == 4 and x.shape[0] == 1) else x),
        DeleteItemsd(keys=["brain_mask_tmp"]),
    ])

# =========================================================
# Section 9: Heavy Augmentation (The "Overfitting-Friendly" Version)
# =========================================================
def build_aug_transform(cfg):
    return Compose([
        RandCropByPosNegLabeld(
            keys=["image", "label", "brain_mask_tmp", "label_bin_tmp"],
            label_key="label_bin_tmp",
            spatial_size=cfg.patch_size,
            pos=2.0, # تمرکز بیشتر روی تومور
            neg=1.0,
            num_samples=getattr(cfg, "num_samples", 1),
            image_key="image",
            image_threshold=0,
        ),
        # آگمنتیشن‌های سنگین برای یادگیری ویژگی‌های دشوار
        RandBiasFieldd(keys=["image"], prob=0.3, coeff_range=(0.2, 0.3)),
        RandFlipd(keys=["image", "label", "brain_mask_tmp"], prob=0.5, spatial_axis=0),
        RandFlipd(keys=["image", "label", "brain_mask_tmp"], prob=0.5, spatial_axis=1),
        RandFlipd(keys=["image", "label", "brain_mask_tmp"], prob=0.5, spatial_axis=2),
        RandRotate90d(keys=["image", "label", "brain_mask_tmp"], prob=0.5, max_k=3),

        RandGaussianNoised(keys=["image"], prob=0.15, mean=0.0, std=0.01),
        RandGaussianSmoothd(keys=["image"], prob=0.2, sigma_x=(0.5, 1.0), sigma_y=(0.5, 1.0), sigma_z=(0.5, 1.0)),

        RandScaleIntensityd(keys=["image"], factors=0.10, prob=0.5),
        RandShiftIntensityd(keys=["image"], offsets=0.10, prob=0.5),
        RandAdjustContrastd(keys=["image"], gamma=(0.5, 4.5), prob=0.5), # گامای قدرتمند شما

        MaskIntensityd(keys=["image"], mask_key="brain_mask_tmp"),

        Lambdad(keys=["label"], func=lambda x: x.long()),
        Lambdad(keys=["label"], func=lambda x: x.squeeze(0) if (x.ndim == 4 and x.shape[0] == 1) else x),

        # پاکسازی نهایی برای جلوگیری از خطای NameError و نشت حافظه
        DeleteItemsd(keys=["brain_mask_tmp", "label_bin_tmp"]),
        EnsureTyped(keys=["image", "label"]),
    ])

# =========================================================
# Section 10: Data Builders (Caching & Loading Strategy) ✅
# =========================================================
import random
import numpy as np
import torch
from pathlib import Path
from monai.data import (
    Dataset as MonaiDataset, CacheDataset, PersistentDataset,
    DataLoader, list_data_collate
)

# ---------------------------------------------------------
# Helper Functions for Reproducibility
# ---------------------------------------------------------
def _seed_worker(worker_id: int):
    """تنظیم سید برای هر ورکر جهت اطمینان از تکرارپذیری"""
    worker_seed = torch.initial_seed() % 2**32
    random.seed(worker_seed)
    np.random.seed(worker_seed)

def _make_generator(seed: int | None):
    """ساخت جنریتور رندوم برای دیتالودر"""
    if seed is None: return None
    g = torch.Generator()
    g.manual_seed(int(seed))
    return g

# ---------------------------------------------------------
# 1. Dataset Builder (Two-Stage Pipeline)
# ---------------------------------------------------------
def build_train_datasets(base_ds, cfg, pre_transform, aug_transform):
    """
    ساخت دیتاست آموزشی با استراتژی کشینگ (Cache Strategy).
    """
    backend = str(getattr(cfg, "cache_backend", "none")).lower()
    cache_rate = float(getattr(cfg, "cache_rate", 0.0))
    nw = int(getattr(cfg, "num_workers", 2))

    # --- مرحله 1: کش کردن دیتای اولیه ---
    # اگر کش ریت 0 باشد، حتی اگر رم انتخاب شده باشد، باید کش غیرفعال شود
    if backend == "ram" and cache_rate > 0:
        print(f"🚀 [Cache] Loading {cache_rate*100}% data into RAM with {nw} workers...")
        pre_ds = CacheDataset(
            data=base_ds,
            transform=pre_transform,
            cache_rate=cache_rate,
            num_workers=nw,
        )
    elif backend == "disk":
        cache_dir = Path(getattr(cfg, "cache_dir", "./cache_pre"))
        cache_dir.mkdir(parents=True, exist_ok=True)
        print(f"💾 [Cache] Using Disk Cache at: {cache_dir}")
        pre_ds = PersistentDataset(
            data=base_ds,
            transform=pre_transform,
            cache_dir=str(cache_dir),
        )
    else:
        # حالت None یا وقتی که cache_rate=0 است
        if backend == "ram": print("⚠️ Cache Rate is 0.0 -> Switching to standard Dataset.")
        pre_ds = MonaiDataset(data=base_ds, transform=pre_transform)

    # --- مرحله 2: اعمال آگمنتیشن‌های تصادفی روی دیتای کش شده ---
    train_ds = MonaiDataset(data=pre_ds, transform=aug_transform)
    return train_ds

# ---------------------------------------------------------
# 2. Train Loader Builder
# ---------------------------------------------------------
def build_train_loader(train_ds, cfg):
    """
    ساخت دیتالودر مخصوص آموزش با list_data_collate
    """
    bs = int(getattr(cfg, "batch_size", 1))
    nw = int(getattr(cfg, "num_workers", 2))
    pin = bool(getattr(cfg, "pin_memory", True))
    seed = getattr(cfg, "seed", None)

    # اگر تعداد ورکرها 0 باشد، persistent_workers باید خاموش باشد
    persistent = (nw > 0)

    return DataLoader(
        dataset=train_ds,
        batch_size=bs,
        shuffle=True,
        num_workers=nw,
        pin_memory=pin,
        drop_last=False,
        # حیاتی: چون خروجی RandCrop یک لیست است، باید با این تابع جمع شود
        collate_fn=list_data_collate,
        persistent_workers=persistent,
        worker_init_fn=_seed_worker,
        generator=_make_generator(seed),
        prefetch_factor=2 if nw > 0 else None
    )

# ---------------------------------------------------------
# 3. Validation Loader Builder
# ---------------------------------------------------------
def build_val_loader(val_base, cfg, pre_val_transform):
    """
    ساخت دیتالودر ولیدیشن (بدون شافل، بچ سایز 1)
    """
    # ولیدیشن آگمنتیشن ندارد، فقط pre_transform
    val_ds = MonaiDataset(data=val_base, transform=pre_val_transform)

    nw = int(getattr(cfg, "num_workers", 2))
    pin = bool(getattr(cfg, "pin_memory", True))
    seed = getattr(cfg, "seed", None)
    persistent = (nw > 0)

    return DataLoader(
        dataset=val_ds,
        batch_size=1, # ولیدیشن همیشه تک‌تصویر است
        shuffle=False,
        num_workers=nw,
        pin_memory=pin,
        persistent_workers=persistent,
        worker_init_fn=_seed_worker,
        generator=_make_generator(seed),
        prefetch_factor=2 if nw > 0 else None
    )

# =========================================================
# Section 11: Loader Builder — CLEAN VERSION ✅
# =========================================================

def build_loaders(cfg, data_root=None):

    if data_root is None:
        data_root = cfg.dataset_path

    print(f"📂 Loading Dataset from: {data_root}")

    # 1. ساخت دیتاست‌های پایه (Passing explicit configs)
    # حتماً kfold و seed را از cfg بگیرید تا تقسیم‌بندی درست باشد
    train_base = BraTSDictDataset(
        dataset_path=data_root,
        training=True,
        kfold=getattr(cfg, "kfold", 5),
        fold=getattr(cfg, "fold", 0),
        seed=getattr(cfg, "seed", 42),
        verbose=True
    )

    val_base = BraTSDictDataset(
        dataset_path=data_root,
        training=False,
        kfold=getattr(cfg, "kfold", 5),
        fold=getattr(cfg, "fold", 0),
        seed=getattr(cfg, "seed", 42),
        verbose=True
    )

    print(f"📊 [Stats] Train: {len(train_base)} | Val: {len(val_base)}")

    # 2. ساخت ترنسفرم‌ها
    print("⚙️ Building Transforms...")
    pre_tr  = build_pre_transform_train(cfg)
    aug_tr  = build_aug_transform(cfg)
    pre_val = build_pre_transform_val(cfg)

    # 3. ترکیب و ساخت لودرها
    print("🚀 Initializing Pipeline...")

    # ساخت دیتاست نهایی (شامل کشینگ در صورت نیاز)
    train_ds = build_train_datasets(train_base, cfg, pre_tr, aug_tr)

    # ساخت دیتالودرها
    train_loader = build_train_loader(train_ds, cfg)
    val_loader = build_val_loader(val_base, cfg, pre_val)

    print("✅ All Loaders Ready!")
    return train_loader, val_loader


cfg = CFG()


#ZIP_PATH   = "/content/data/BraTS/BraTS2021_Training_Data.tar"

ZIP_PATH   = "/content/drive/My Drive/brats2021_20.zip"

EXTRACT_TO = "/content/data/BraTS"      # این‌جا بدون تودرتو می‌ریزه
train_cache  = "/content/cache_pre_train"
val_cache    = "/content/cache_pre_val"

dataset_root = prepare_dataset_from_archive(
    cfg,
    archive_path=ZIP_PATH,
    extract_dir=EXTRACT_TO,
    ensure_cache_dirs=(train_cache, val_cache),
    force_reextract=True,
    )



cfg.dataset_path = dataset_root
print(cfg.dataset_path)
'''

In [ ]:
'''# smoother aug

import torch.nn.functional as F
import os, tarfile, zipfile, json, time
from typing import Optional, Tuple

def _human(n):
    for u in ["B","KB","MB","GB","TB"]:
        if n < 1024: return f"{n:.1f} {u}"
        n /= 1024
    return f"{n:.1f} PB"

def prepare_dataset_from_archive(
    cfg,
    archive_path: str,
    extract_dir: str,
    ensure_cache_dirs: Optional[Tuple[str, str]] = None,
    force_reextract: bool = False,
) -> str:
    """ نسخه آپدیت شده: پشتیبانی کامل از TAR و ZIP """
    if not os.path.exists(archive_path):
        raise FileNotFoundError(f"Archive not found: {archive_path}")

    os.makedirs(extract_dir, exist_ok=True)

    # اگر پوشه پر است و فورس نکردیم، همان را برگردان
    if not force_reextract and len(os.listdir(extract_dir)) > 0:
        print(f"[dataset] Found existing data in {extract_dir}")
        # پیدا کردن پوشه داخلی
        subs = [d for d in os.listdir(extract_dir) if os.path.isdir(os.path.join(extract_dir, d))]
        return os.path.join(extract_dir, subs[0]) if len(subs) == 1 else extract_dir

    print(f"[dataset] extracting: {archive_path} -> {extract_dir}")

    if archive_path.endswith(".tar"):
        with tarfile.open(archive_path, "r") as tar:
            tar.extractall(path=extract_dir)
    elif archive_path.endswith(".zip"):
        with zipfile.ZipFile(archive_path, "r") as z:
            z.extractall(extract_dir)
    else:
        raise ValueError("Format not supported (only .tar or .zip)")

    print("Done.")

    # پیدا کردن پوشه ریشه دیتاست
    subs = [d for d in os.listdir(extract_dir) if os.path.isdir(os.path.join(extract_dir, d))]
    final_root = os.path.join(extract_dir, subs[0]) if len(subs) == 1 else extract_dir
    return final_root
   # section 1
import os, glob
import numpy as np
import nibabel as nib

def _load_nii(path, to_float=True, canonical=True):
    img = nib.load(path)
    if canonical:
        img = nib.as_closest_canonical(img)  # RAS
    arr = img.get_fdata(dtype=np.float32 if to_float else None)
    return arr

def _find_one(folder, patterns):
    """
    اولین فایلی که با یکی از الگوها match شود را برمی‌گرداند.
    الگوها باید شامل wildcard باشند (مثلاً *.nii* تا .nii و .nii.gz را بگیرد).
    """
    for p in patterns:
        m = sorted(glob.glob(os.path.join(folder, p)))
        if m:
            return m[0]
    raise FileNotFoundError(f"Not found in {folder}: {patterns}")

def load_modalities_from_folder(folder_name: str, image_id: str = ""):

    """
    خروجی:
      image: (4, W, H,  D) ترتیب [T1, T1Gd/T1ce, T2, FLAIR] (float32)
      mask : ( W, H, D) (int64)
    - به‌صورت محافظه‌کارانه، الگوها چندین نام رایج را پوشش می‌دهند.
    - اگر image_id ندادی، باز هم با الگوهای عمومی کار می‌کند.
    """
    # ---- الگوهای قابل‌قبول برای هر مودالیتی ----
    # توجه: ترتیب الگوها از خاص به عام است تا اشتباه match نشود.
    t1_patterns = [
        f"*{image_id}*t1.nii*", "*_t1.nii*", "*t1w.nii*", "*t1w*.nii*", "*t1n*.nii*"
    ]
    t1ce_patterns = [
        f"*{image_id}*t1ce.nii*", "*t1ce.nii*", "*t1c.nii*", "*t1gd.nii*", "*t1Gd.nii*"
    ]
    t2_patterns = [
        f"*{image_id}*t2.nii*", "*_t2.nii*", "*t2w.nii*"
    ]
    flair_patterns = [
        f"*{image_id}*flair.nii*", "*_flair.nii*", "*t2flair.nii*", "*t2f.nii*"
    ]
    seg_patterns = [
        f"*{image_id}*seg.nii*", "*_seg.nii*", "*segmentation*.nii*"
    ]

    # ---- پیدا کردن مسیر فایل‌ها ----
    t1n = _find_one(folder_name, t1_patterns)
    t1c = _find_one(folder_name, t1ce_patterns)
    t2w = _find_one(folder_name, t2_patterns)
    t2f = _find_one(folder_name, flair_patterns)
    seg = _find_one(folder_name, seg_patterns)

    # ---- لود داده‌ها ----
    a_t1  = _load_nii(t1n,  to_float=True, canonical=True)   # (w,h,d)
    a_t1c = _load_nii(t1c,  to_float=True, canonical=True)
    a_t2  = _load_nii(t2w,  to_float=True, canonical=True)
    a_fl  = _load_nii(t2f,  to_float=True, canonical=True)
    m_seg = _load_nii(seg,  to_float=False, canonical=True ).astype(np.int64)  # (W,H,D)


    # ---- sanity check شکل‌ها (فقط اخطار) ----
    shapes = {tuple(a_t1.shape), tuple(a_t1c.shape), tuple(a_t2.shape), tuple(a_fl.shape), tuple(m_seg.shape)}
    if len(shapes) != 1:
        print(f"[warn] mismatched shapes in {folder_name}: {shapes}")

    # ---- استک کانال-اول ----
    image = np.stack([a_t1, a_t1c, a_t2, a_fl], axis=0).astype(np.float32)  # (4,W,H,D)
    mask  = m_seg  # (W, H, D)  int64

    #print('         ------   loaded_modalities_from_folder ✅  ---------    ')

    return image, mask
# section 2
import numpy as np

def remap_label_4to3(mask: np.ndarray) -> np.ndarray:
    """
    BraTS: کلاس 4 → 3 (ET)
    ورودی: (H,W,D) عددی
    خروجی: (H,W,D) int64
    """
    out = mask.astype(np.int64, copy=True)
    out[out == 4] = 3
    return out

def center_pad_crop(arr: np.ndarray, out_shape, pad_val=0.0):
    """
    آرایه را به out_shape به‌صورت مرکز-محور پد/کراپ می‌کند.
    با ترتیب (W, H, D) سازگار است.
    """
    # تشخیص تعداد کانال‌ها
    if arr.ndim == 4:
        # ورودی: (C, W, H, D)
        C, W, H, D = arr.shape
        tgt_W, tgt_H, tgt_D = out_shape
        out = np.full((C, tgt_W, tgt_H, tgt_D), pad_val, dtype=arr.dtype)
    elif arr.ndim == 3:
        # ورودی: (W, H, D)
        W, H, D = arr.shape
        tgt_W, tgt_H, tgt_D = out_shape
        out = np.full((tgt_W, tgt_H, tgt_D), pad_val, dtype=arr.dtype)
    else:
        raise ValueError("آرایه باید 3 یا 4 بعدی باشد.")

    def get_sl(src_len, dst_len):
        if src_len >= dst_len:
            # Crop
            start = (src_len - dst_len) // 2
            return slice(start, start + dst_len), slice(0, dst_len)
        else:
            # Pad
            start = (dst_len - src_len) // 2
            return slice(0, src_len), slice(start, start + src_len)

    # محاسبه اسلایس‌ها برای هر بعد
    sW, dW = get_sl(W, tgt_W)
    sH, dH = get_sl(H, tgt_H)
    sD, dD = get_sl(D, tgt_D)

    if arr.ndim == 4:
        out[:, dW, dH, dD] = arr[:, sW, sH, sD]
    else:
        out[dW, dH, dD] = arr[sW, sH, sD]

    return out

def make_brain_mask(image: np.ndarray, pad_val=0.0) -> np.ndarray:
    """
    mask باینری مغز از روی خود تصویر:
      - هر وکسل که همه‌ی کانال‌هاش == pad_val باشد، خارج مغز است.
      - بقیه مغز.
    ورودی: image شکل (C, W, H, D)
    خروجی: (W, H, D) از نوع bool

    """
    assert image.ndim == 4
    return np.any(image != pad_val, axis=0)


# =========================================================
# Section 3: Dataset Class (BraTS Dict Dataset) — FIXED 💎
# =========================================================
import os, random
import numpy as np
import torch
from torch.utils.data import Dataset
from monai.data import MetaTensor

def list_case_folders(dataset_path):
    return [d for d in sorted(os.listdir(dataset_path))
            if os.path.isdir(os.path.join(dataset_path, d))]

def create_case_splits(dataset_path, kfold=5, fold=0, seed=42):
    cases_all = list_case_folders(dataset_path)
    rng = random.Random(seed)
    rng.shuffle(cases_all)
    idx = np.arange(len(cases_all))
    folds = np.array_split(idx, kfold)
    val_idx = set(folds[fold])
    train_cases = [cases_all[i] for i in idx if i not in val_idx]
    val_cases   = [cases_all[i] for i in idx if i in val_idx]
    return train_cases, val_cases

class BraTSDictDataset(Dataset):
    def __init__(self, dataset_path, training=True, kfold=5, fold=0,
                 out_shape=None, remap_4to3=True, pad_val=0.0, seed=42, verbose=False):
        super().__init__()
        self.dataset_path = dataset_path
        self.training = training
        self.out_shape = out_shape
        self.remap_4to3 = remap_4to3
        self.pad_val = pad_val
        self.verbose = verbose

        train_cases, val_cases = create_case_splits(dataset_path, kfold=kfold, fold=fold, seed=seed)
        self.cases = train_cases if training else val_cases

        if verbose:
            split = "train" if training else "val"
            print(f"[{split}] cases={len(self.cases)}")

    def __len__(self):
        return len(self.cases)

    def __getitem__(self, idx):
        case_id = self.cases[idx]
        folder = os.path.join(self.dataset_path, case_id)

        # img: (4, W, H, D)
        # lbl: (W, H, D)
        img, lbl = load_modalities_from_folder(folder_name=folder, image_id=case_id)

        if self.remap_4to3:
            lbl = remap_label_4to3(lbl)

        if self.out_shape is not None:
            img = center_pad_crop(img, self.out_shape, pad_val=self.pad_val)
            lbl = center_pad_crop(lbl, self.out_shape, pad_val=0)

        brain_mask = make_brain_mask(img, pad_val=self.pad_val)

        # --- 🔴 اصلاح حیاتی: اضافه کردن بعد کانال به لیبل ---
        # لیبل از (W,H,D) به (1,W,H,D) تبدیل می‌شود تا Spacingd گیج نشود.
        lbl = lbl[None, ...]

        # تبدیل به MetaTensor برای حفظ اطلاعات ابعاد
        img_t = MetaTensor(torch.from_numpy(img.astype(np.float32)), affine=torch.eye(4))
        lbl_t = MetaTensor(torch.from_numpy(lbl.astype(np.float32)), affine=torch.eye(4)) # موقتا فلوت برای اینترپولیشن
        bm_t  = torch.from_numpy(brain_mask.astype(bool))

        return {"image": img_t, "label": lbl_t, "brain_mask": bm_t, "case_id": case_id}

# section 4

def mk_brain_mask_lambdad(cfg):
    tol = float(getattr(cfg, "bg_tol", 1e-6))
    pad_val = float(getattr(cfg, "pad_val", 0.0))

    def _f(im):
        # تبدیل به float برای محاسبات دقیق
        im32 = im.float()

        # ۱. تشخیص نقاطی که پدینگ یا صفر مطلق نیستند
        # نکته: استفاده از im32.abs() > 1e-4 برای اطمینان از حذف نویزهای میکروسکوپی
        m_pad = (im32 != pad_val) & (torch.abs(im32) > tol)

        # ۲. اگر در هر یک از ۴ مدالیته (T1, T2, ...) سیگنالی وجود داشته باشد، آنجا را مغز فرض کن
        m = m_pad.any(dim=0)

        # ۳. (اختیاری اما پیشنهادی) پر کردن سوراخ‌های احتمالی داخل ماسک
        # اگر در آینده متوجه شدی داخل تومورها "سیاه" می‌افتد، اینجا می‌توان از
        # عملیات مورفولوژیک استفاده کرد. اما برای شروع همین کافی است.

        return m.unsqueeze(0).float() # خروجی (1, W, H, D) به صورت 0 و 1

    return Lambdad(keys=["brain_mask_tmp"], func=_f)
# section 5

def dilate_label_bin_lambdad(kernel: int = 7):
    assert kernel % 2 == 1, "kernel باید فرد باشد."
    pad = kernel // 2

    def _f(x):
        # x شکلش هست (1, W, H, D)
        if x.ndim == 3:
            x = x.unsqueeze(0)

        # اطمینان از باینری بودن و نوع داده
        x = (x > 0).float()

        # بردن به فرمت (N, C, D, H, W) که برای max_pool3d بهینه است
        # نکته: ترتیب W,H,D مهم نیست چون کرنل متقارن است، فقط باید 5 بعدی شود
        xb = x.unsqueeze(0) # (1, 1, W, H, D)

        # عملیات دیلیت (بزرگتر کردن ناحیه تومور)
        y = F.max_pool3d(xb, kernel_size=kernel, stride=1, padding=pad)

        # برگشت به شکل (1, W, H, D) و تبدیل به uint8 برای اشغال حافظه کمتر
        return y.squeeze(0).to(torch.uint8)

    return Lambdad(keys=["label_bin_tmp"], func=_f)
# =========================================================
# Section 6: Pre-Processing Transforms (TRAIN) — FIXED 💎
# =========================================================
from monai.transforms import (
    Compose, EnsureTyped, Orientationd, Spacingd, Lambdad,
    CropForegroundd, CopyItemsd, MaskIntensityd,
    NormalizeIntensityd, SpatialPadd
)
# =========================================================
# Section 6: Pre-Processing Transforms (TRAIN) — UPDATED 💎
# =========================================================
def build_pre_transform_train(cfg):
    return Compose([
        EnsureTyped(keys=["image", "label"], track_meta=True),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        Spacingd(
            keys=["image", "label"],
            pixdim=(1.0, 1.0, 1.0),
            mode=("bilinear", "nearest")
        ),
        CropForegroundd(keys=["image", "label"], source_key="image"),

        # --- اصلاح شده: نرمال‌سازی استاندارد بدون ماسک اضافه ---
        NormalizeIntensityd(keys=["image"], channel_wise=True, nonzero=True),

        SpatialPadd(keys=["image", "label"], spatial_size=cfg.patch_size),
        CopyItemsd(keys=["label"], names=["label_bin_tmp"]),
        Lambdad(keys=["label_bin_tmp"], func=lambda x: (x > 0).float()),
        dilate_label_bin_lambdad(kernel=getattr(cfg, "pos_dilate_kernel", 7)),
    ])

# =========================================================
# Section 7: Pre-Processing Transforms (VAL) — UPDATED ✅
# =========================================================
def build_pre_transform_val(cfg):
    return Compose([
        EnsureTyped(keys=["image", "label"], track_meta=True),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        Spacingd(
            keys=["image", "label"],
            pixdim=(1.0, 1.0, 1.0),
            mode=("bilinear", "nearest")
        ),
        Lambdad(keys=["label"], func=lambda x: x.unsqueeze(0) if x.ndim == 3 else x),
        CropForegroundd(keys=["image", "label"], source_key="image"),

        # --- اصلاح شده: نرمال‌سازی مشابه Train ---
        NormalizeIntensityd(keys=["image"], channel_wise=True, nonzero=True),

        SpatialPadd(keys=["image", "label"], spatial_size=cfg.patch_size),
        Lambdad(keys=["label"], func=lambda x: x.long()),
        Lambdad(keys=["label"], func=lambda x: x.squeeze(0) if (x.ndim == 4 and x.shape[0] == 1) else x),
    ])

# =========================================================
# Section 9: Heavy Augmentation Transform — SAFE ✅
# =========================================================
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    EnsureTyped,
    Orientationd,
    Spacingd,
    NormalizeIntensityd,
    CastToTyped,

    # ---------- Cropping / Sampling ----------
    RandCropByPosNegLabeld,
    RandSpatialCropd,

    # ---------- Intensity Augmentations ----------
    RandScaleIntensityd,
    RandShiftIntensityd,
    RandGaussianNoised,
    RandGaussianSmoothd,
    RandBiasFieldd,
    RandAdjustContrastd,   # ✅ این یکی برای ارور فعلی

    # ---------- Spatial Augmentations ----------
    RandFlipd,
    RandAffined,
    RandRotate90d,

    # ---------- Masking ----------
    MaskIntensityd,        # چون از brain_mask_tmp استفاده می‌کنی
)


# =========================================================
# Section 9: Heavy Augmentation Transform — FIXED 💎
# =========================================================
from monai.transforms import (
    Compose, EnsureTyped, RandFlipd, RandRotate90d,
    RandGaussianNoised, RandGaussianSmoothd, RandScaleIntensityd,
    RandShiftIntensityd, RandAdjustContrastd, RandBiasFieldd,
    RandCropByPosNegLabeld, Lambdad, DeleteItemsd  # <--- DeleteItemsd اضافه شد
)

def build_aug_transform(cfg):
    return Compose([
        # کراپ هوشمند روی نواحی تومورال
        RandCropByPosNegLabeld(
            keys=["image", "label", "label_bin_tmp"],
            label_key="label_bin_tmp",
            spatial_size=cfg.patch_size,
            pos=2.0,
            neg=1.0,
            num_samples=getattr(cfg, "num_samples", 1),
            image_key="image",
            image_threshold=0,
        ),

        # آگمنتیشن‌های فضایی
        RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
        RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
        RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=2),
        RandRotate90d(keys=["image", "label"], prob=0.5, max_k=3),

        # آگمنتیشن‌های شدت روشنایی (ملایم شده برای ۲۰ تصویر)
        RandGaussianNoised(keys=["image"], prob=0.15, mean=0.0, std=0.01),
        RandBiasFieldd(keys=["image"], prob=0.2, coeff_range=(0.1, 0.2)),
        RandAdjustContrastd(keys=["image"], gamma=(0.7, 1.5), prob=0.3), # گاما اصلاح شد

        RandScaleIntensityd(keys=["image"], factors=0.10, prob=0.5),
        RandShiftIntensityd(keys=["image"], offsets=0.10, prob=0.5),

        # فرمت نهایی لیبل
        Lambdad(keys=["label"], func=lambda x: x.long()),
        Lambdad(keys=["label"], func=lambda x: x.squeeze(0) if (x.ndim == 4 and x.shape[0] == 1) else x),

        # پاکسازی متغیرهای کمکی برای جلوگیری از اشغال VRAM
        DeleteItemsd(keys=["label_bin_tmp"]),
        EnsureTyped(keys=["image", "label"]),
    ])

# =========================================================
# Section 10: Data Builders (Caching & Loading Strategy) ✅
# =========================================================
import random
import numpy as np
import torch
from pathlib import Path
from monai.data import (
    Dataset as MonaiDataset, CacheDataset, PersistentDataset,
    DataLoader, list_data_collate
)

# ---------------------------------------------------------
# Helper Functions for Reproducibility
# ---------------------------------------------------------
def _seed_worker(worker_id: int):
    """تنظیم سید برای هر ورکر جهت اطمینان از تکرارپذیری"""
    worker_seed = torch.initial_seed() % 2**32
    random.seed(worker_seed)
    np.random.seed(worker_seed)

def _make_generator(seed: int | None):
    """ساخت جنریتور رندوم برای دیتالودر"""
    if seed is None: return None
    g = torch.Generator()
    g.manual_seed(int(seed))
    return g

# ---------------------------------------------------------
# 1. Dataset Builder (Two-Stage Pipeline)
# ---------------------------------------------------------
def build_train_datasets(base_ds, cfg, pre_transform, aug_transform):
    """
    ساخت دیتاست آموزشی با استراتژی کشینگ (Cache Strategy).
    """
    backend = str(getattr(cfg, "cache_backend", "none")).lower()
    cache_rate = float(getattr(cfg, "cache_rate", 0.0))
    nw = int(getattr(cfg, "num_workers", 2))

    # --- مرحله 1: کش کردن دیتای اولیه ---
    # اگر کش ریت 0 باشد، حتی اگر رم انتخاب شده باشد، باید کش غیرفعال شود
    if backend == "ram" and cache_rate > 0:
        print(f"🚀 [Cache] Loading {cache_rate*100}% data into RAM with {nw} workers...")
        pre_ds = CacheDataset(
            data=base_ds,
            transform=pre_transform,
            cache_rate=cache_rate,
            num_workers=nw,
        )
    elif backend == "disk":
        cache_dir = Path(getattr(cfg, "cache_dir", "./cache_pre"))
        cache_dir.mkdir(parents=True, exist_ok=True)
        print(f"💾 [Cache] Using Disk Cache at: {cache_dir}")
        pre_ds = PersistentDataset(
            data=base_ds,
            transform=pre_transform,
            cache_dir=str(cache_dir),
        )
    else:
        # حالت None یا وقتی که cache_rate=0 است
        if backend == "ram": print("⚠️ Cache Rate is 0.0 -> Switching to standard Dataset.")
        pre_ds = MonaiDataset(data=base_ds, transform=pre_transform)

    # --- مرحله 2: اعمال آگمنتیشن‌های تصادفی روی دیتای کش شده ---
    train_ds = MonaiDataset(data=pre_ds, transform=aug_transform)
    return train_ds

# ---------------------------------------------------------
# 2. Train Loader Builder
# ---------------------------------------------------------
def build_train_loader(train_ds, cfg):
    """
    ساخت دیتالودر مخصوص آموزش با list_data_collate
    """
    bs = int(getattr(cfg, "batch_size", 1))
    nw = int(getattr(cfg, "num_workers", 2))
    pin = bool(getattr(cfg, "pin_memory", True))
    seed = getattr(cfg, "seed", None)

    # اگر تعداد ورکرها 0 باشد، persistent_workers باید خاموش باشد
    persistent = (nw > 0)

    return DataLoader(
        dataset=train_ds,
        batch_size=bs,
        shuffle=True,
        num_workers=nw,
        pin_memory=pin,
        drop_last=False,
        # حیاتی: چون خروجی RandCrop یک لیست است، باید با این تابع جمع شود
        collate_fn=list_data_collate,
        persistent_workers=persistent,
        worker_init_fn=_seed_worker,
        generator=_make_generator(seed),
        prefetch_factor=2 if nw > 0 else None
    )

# ---------------------------------------------------------
# 3. Validation Loader Builder
# ---------------------------------------------------------
def build_val_loader(val_base, cfg, pre_val_transform):
    """
    ساخت دیتالودر ولیدیشن (بدون شافل، بچ سایز 1)
    """
    # ولیدیشن آگمنتیشن ندارد، فقط pre_transform
    val_ds = MonaiDataset(data=val_base, transform=pre_val_transform)

    nw = int(getattr(cfg, "num_workers", 2))
    pin = bool(getattr(cfg, "pin_memory", True))
    seed = getattr(cfg, "seed", None)
    persistent = (nw > 0)

    return DataLoader(
        dataset=val_ds,
        batch_size=1, # ولیدیشن همیشه تک‌تصویر است
        shuffle=False,
        num_workers=nw,
        pin_memory=pin,
        persistent_workers=persistent,
        worker_init_fn=_seed_worker,
        generator=_make_generator(seed),
        prefetch_factor=2 if nw > 0 else None
    )

# =========================================================
# Section 11: Loader Builder — CLEAN VERSION ✅
# =========================================================

def build_loaders(cfg, data_root=None):

    if data_root is None:
        data_root = cfg.dataset_path

    print(f"📂 Loading Dataset from: {data_root}")

    # 1. ساخت دیتاست‌های پایه (Passing explicit configs)
    # حتماً kfold و seed را از cfg بگیرید تا تقسیم‌بندی درست باشد
    train_base = BraTSDictDataset(
        dataset_path=data_root,
        training=True,
        kfold=getattr(cfg, "kfold", 5),
        fold=getattr(cfg, "fold", 0),
        seed=getattr(cfg, "seed", 42),
        verbose=True
    )

    val_base = BraTSDictDataset(
        dataset_path=data_root,
        training=False,
        kfold=getattr(cfg, "kfold", 5),
        fold=getattr(cfg, "fold", 0),
        seed=getattr(cfg, "seed", 42),
        verbose=True
    )

    print(f"📊 [Stats] Train: {len(train_base)} | Val: {len(val_base)}")

    # 2. ساخت ترنسفرم‌ها
    print("⚙️ Building Transforms...")
    pre_tr  = build_pre_transform_train(cfg)
    aug_tr  = build_aug_transform(cfg)
    pre_val = build_pre_transform_val(cfg)

    # 3. ترکیب و ساخت لودرها
    print("🚀 Initializing Pipeline...")

    # ساخت دیتاست نهایی (شامل کشینگ در صورت نیاز)
    train_ds = build_train_datasets(train_base, cfg, pre_tr, aug_tr)

    # ساخت دیتالودرها
    train_loader = build_train_loader(train_ds, cfg)
    val_loader = build_val_loader(val_base, cfg, pre_val)

    print("✅ All Loaders Ready!")
    return train_loader, val_loader


cfg = CFG()


#ZIP_PATH   = "/content/data/BraTS/BraTS2021_Training_Data.tar"

ZIP_PATH   = "/content/drive/My Drive/brats2021_20.zip"

EXTRACT_TO = "/content/data/BraTS"      # این‌جا بدون تودرتو می‌ریزه
train_cache  = "/content/cache_pre_train"
val_cache    = "/content/cache_pre_val"

dataset_root = prepare_dataset_from_archive(
    cfg,
    archive_path=ZIP_PATH,
    extract_dir=EXTRACT_TO,
    ensure_cache_dirs=(train_cache, val_cache),
    force_reextract=True,
    )



cfg.dataset_path = dataset_root
print(cfg.dataset_path)
'''

In [ ]:
#original
'''
import torch.nn.functional as F
import os, tarfile, zipfile, json, time
from typing import Optional, Tuple

def _human(n):
    for u in ["B","KB","MB","GB","TB"]:
        if n < 1024: return f"{n:.1f} {u}"
        n /= 1024
    return f"{n:.1f} PB"

def prepare_dataset_from_archive(
    cfg,
    archive_path: str,
    extract_dir: str,
    ensure_cache_dirs: Optional[Tuple[str, str]] = None,
    force_reextract: bool = False,
) -> str:
    """ نسخه آپدیت شده: پشتیبانی کامل از TAR و ZIP """
    if not os.path.exists(archive_path):
        raise FileNotFoundError(f"Archive not found: {archive_path}")

    os.makedirs(extract_dir, exist_ok=True)

    # اگر پوشه پر است و فورس نکردیم، همان را برگردان
    if not force_reextract and len(os.listdir(extract_dir)) > 0:
        print(f"[dataset] Found existing data in {extract_dir}")
        # پیدا کردن پوشه داخلی
        subs = [d for d in os.listdir(extract_dir) if os.path.isdir(os.path.join(extract_dir, d))]
        return os.path.join(extract_dir, subs[0]) if len(subs) == 1 else extract_dir

    print(f"[dataset] extracting: {archive_path} -> {extract_dir}")

    if archive_path.endswith(".tar"):
        with tarfile.open(archive_path, "r") as tar:
            tar.extractall(path=extract_dir)
    elif archive_path.endswith(".zip"):
        with zipfile.ZipFile(archive_path, "r") as z:
            z.extractall(extract_dir)
    else:
        raise ValueError("Format not supported (only .tar or .zip)")

    print("Done.")

    # پیدا کردن پوشه ریشه دیتاست
    subs = [d for d in os.listdir(extract_dir) if os.path.isdir(os.path.join(extract_dir, d))]
    final_root = os.path.join(extract_dir, subs[0]) if len(subs) == 1 else extract_dir
    return final_root
   # section 1
import os, glob
import numpy as np
import nibabel as nib

def _load_nii(path, to_float=True, canonical=True):
    img = nib.load(path)
    if canonical:
        img = nib.as_closest_canonical(img)  # RAS
    arr = img.get_fdata(dtype=np.float32 if to_float else None)
    return arr

def _find_one(folder, patterns):
    """
    اولین فایلی که با یکی از الگوها match شود را برمی‌گرداند.
    الگوها باید شامل wildcard باشند (مثلاً *.nii* تا .nii و .nii.gz را بگیرد).
    """
    for p in patterns:
        m = sorted(glob.glob(os.path.join(folder, p)))
        if m:
            return m[0]
    raise FileNotFoundError(f"Not found in {folder}: {patterns}")

def load_modalities_from_folder(folder_name: str, image_id: str = ""):

    """
    خروجی:
      image: (4, W, H,  D) ترتیب [T1, T1Gd/T1ce, T2, FLAIR] (float32)
      mask : ( W, H, D) (int64)
    - به‌صورت محافظه‌کارانه، الگوها چندین نام رایج را پوشش می‌دهند.
    - اگر image_id ندادی، باز هم با الگوهای عمومی کار می‌کند.
    """
    # ---- الگوهای قابل‌قبول برای هر مودالیتی ----
    # توجه: ترتیب الگوها از خاص به عام است تا اشتباه match نشود.
    t1_patterns = [
        f"*{image_id}*t1.nii*", "*_t1.nii*", "*t1w.nii*", "*t1w*.nii*", "*t1n*.nii*"
    ]
    t1ce_patterns = [
        f"*{image_id}*t1ce.nii*", "*t1ce.nii*", "*t1c.nii*", "*t1gd.nii*", "*t1Gd.nii*"
    ]
    t2_patterns = [
        f"*{image_id}*t2.nii*", "*_t2.nii*", "*t2w.nii*"
    ]
    flair_patterns = [
        f"*{image_id}*flair.nii*", "*_flair.nii*", "*t2flair.nii*", "*t2f.nii*"
    ]
    seg_patterns = [
        f"*{image_id}*seg.nii*", "*_seg.nii*", "*segmentation*.nii*"
    ]

    # ---- پیدا کردن مسیر فایل‌ها ----
    t1n = _find_one(folder_name, t1_patterns)
    t1c = _find_one(folder_name, t1ce_patterns)
    t2w = _find_one(folder_name, t2_patterns)
    t2f = _find_one(folder_name, flair_patterns)
    seg = _find_one(folder_name, seg_patterns)

    # ---- لود داده‌ها ----
    a_t1  = _load_nii(t1n,  to_float=True, canonical=True)   # (w,h,d)
    a_t1c = _load_nii(t1c,  to_float=True, canonical=True)
    a_t2  = _load_nii(t2w,  to_float=True, canonical=True)
    a_fl  = _load_nii(t2f,  to_float=True, canonical=True)
    m_seg = _load_nii(seg,  to_float=False, canonical=True ).astype(np.int64)  # (W,H,D)


    # ---- sanity check شکل‌ها (فقط اخطار) ----
    shapes = {tuple(a_t1.shape), tuple(a_t1c.shape), tuple(a_t2.shape), tuple(a_fl.shape), tuple(m_seg.shape)}
    if len(shapes) != 1:
        print(f"[warn] mismatched shapes in {folder_name}: {shapes}")

    # ---- استک کانال-اول ----
    image = np.stack([a_t1, a_t1c, a_t2, a_fl], axis=0).astype(np.float32)  # (4,W,H,D)
    mask  = m_seg  # (W, H, D)  int64

    #print('         ------   loaded_modalities_from_folder ✅  ---------    ')

    return image, mask
# section 2
import numpy as np

def remap_label_4to3(mask: np.ndarray) -> np.ndarray:
    """
    BraTS: کلاس 4 → 3 (ET)
    ورودی: (H,W,D) عددی
    خروجی: (H,W,D) int64
    """
    out = mask.astype(np.int64, copy=True)
    out[out == 4] = 3
    return out

def center_pad_crop(arr: np.ndarray, out_shape, pad_val=0.0):
    """
    آرایه را به out_shape به‌صورت مرکز-محور پد/کراپ می‌کند.
    با ترتیب (W, H, D) سازگار است.
    """
    # تشخیص تعداد کانال‌ها
    if arr.ndim == 4:
        # ورودی: (C, W, H, D)
        C, W, H, D = arr.shape
        tgt_W, tgt_H, tgt_D = out_shape
        out = np.full((C, tgt_W, tgt_H, tgt_D), pad_val, dtype=arr.dtype)
    elif arr.ndim == 3:
        # ورودی: (W, H, D)
        W, H, D = arr.shape
        tgt_W, tgt_H, tgt_D = out_shape
        out = np.full((tgt_W, tgt_H, tgt_D), pad_val, dtype=arr.dtype)
    else:
        raise ValueError("آرایه باید 3 یا 4 بعدی باشد.")

    def get_sl(src_len, dst_len):
        if src_len >= dst_len:
            # Crop
            start = (src_len - dst_len) // 2
            return slice(start, start + dst_len), slice(0, dst_len)
        else:
            # Pad
            start = (dst_len - src_len) // 2
            return slice(0, src_len), slice(start, start + src_len)

    # محاسبه اسلایس‌ها برای هر بعد
    sW, dW = get_sl(W, tgt_W)
    sH, dH = get_sl(H, tgt_H)
    sD, dD = get_sl(D, tgt_D)

    if arr.ndim == 4:
        out[:, dW, dH, dD] = arr[:, sW, sH, sD]
    else:
        out[dW, dH, dD] = arr[sW, sH, sD]

    return out

def make_brain_mask(image: np.ndarray, pad_val=0.0) -> np.ndarray:
    """
    mask باینری مغز از روی خود تصویر:
      - هر وکسل که همه‌ی کانال‌هاش == pad_val باشد، خارج مغز است.
      - بقیه مغز.
    ورودی: image شکل (C, W, H, D)
    خروجی: (W, H, D) از نوع bool

    """
    assert image.ndim == 4
    return np.any(image != pad_val, axis=0)


# =========================================================
# Section 3: Dataset Class (BraTS Dict Dataset) — FIXED 💎
# =========================================================
import os, random
import numpy as np
import torch
from torch.utils.data import Dataset
from monai.data import MetaTensor

def list_case_folders(dataset_path):
    return [d for d in sorted(os.listdir(dataset_path))
            if os.path.isdir(os.path.join(dataset_path, d))]

def create_case_splits(dataset_path, kfold=5, fold=0, seed=42):
    cases_all = list_case_folders(dataset_path)
    rng = random.Random(seed)
    rng.shuffle(cases_all)
    idx = np.arange(len(cases_all))
    folds = np.array_split(idx, kfold)
    val_idx = set(folds[fold])
    train_cases = [cases_all[i] for i in idx if i not in val_idx]
    val_cases   = [cases_all[i] for i in idx if i in val_idx]
    return train_cases, val_cases

class BraTSDictDataset(Dataset):
    def __init__(self, dataset_path, training=True, kfold=5, fold=0,
                 out_shape=None, remap_4to3=True, pad_val=0.0, seed=42, verbose=False):
        super().__init__()
        self.dataset_path = dataset_path
        self.training = training
        self.out_shape = out_shape
        self.remap_4to3 = remap_4to3
        self.pad_val = pad_val
        self.verbose = verbose

        train_cases, val_cases = create_case_splits(dataset_path, kfold=kfold, fold=fold, seed=seed)
        self.cases = train_cases if training else val_cases

        if verbose:
            split = "train" if training else "val"
            print(f"[{split}] cases={len(self.cases)}")

    def __len__(self):
        return len(self.cases)

    def __getitem__(self, idx):
        case_id = self.cases[idx]
        folder = os.path.join(self.dataset_path, case_id)

        # img: (4, W, H, D)
        # lbl: (W, H, D)
        img, lbl = load_modalities_from_folder(folder_name=folder, image_id=case_id)

        if self.remap_4to3:
            lbl = remap_label_4to3(lbl)

        if self.out_shape is not None:
            img = center_pad_crop(img, self.out_shape, pad_val=self.pad_val)
            lbl = center_pad_crop(lbl, self.out_shape, pad_val=0)

        brain_mask = make_brain_mask(img, pad_val=self.pad_val)

        # --- 🔴 اصلاح حیاتی: اضافه کردن بعد کانال به لیبل ---
        # لیبل از (W,H,D) به (1,W,H,D) تبدیل می‌شود تا Spacingd گیج نشود.
        lbl = lbl[None, ...]

        # تبدیل به MetaTensor برای حفظ اطلاعات ابعاد
        img_t = MetaTensor(torch.from_numpy(img.astype(np.float32)), affine=torch.eye(4))
        lbl_t = MetaTensor(torch.from_numpy(lbl.astype(np.float32)), affine=torch.eye(4)) # موقتا فلوت برای اینترپولیشن
        bm_t  = torch.from_numpy(brain_mask.astype(bool))

        return {"image": img_t, "label": lbl_t, "brain_mask": bm_t, "case_id": case_id}

# section 4

def mk_brain_mask_lambdad(cfg):
    tol = float(getattr(cfg, "bg_tol", 1e-6))
    pad_val = float(getattr(cfg, "pad_val", 0.0))

    def _f(im):
        # تبدیل به float برای محاسبات دقیق
        im32 = im.float()

        # ۱. تشخیص نقاطی که پدینگ یا صفر مطلق نیستند
        # نکته: استفاده از im32.abs() > 1e-4 برای اطمینان از حذف نویزهای میکروسکوپی
        m_pad = (im32 != pad_val) & (torch.abs(im32) > tol)

        # ۲. اگر در هر یک از ۴ مدالیته (T1, T2, ...) سیگنالی وجود داشته باشد، آنجا را مغز فرض کن
        m = m_pad.any(dim=0)

        # ۳. (اختیاری اما پیشنهادی) پر کردن سوراخ‌های احتمالی داخل ماسک
        # اگر در آینده متوجه شدی داخل تومورها "سیاه" می‌افتد، اینجا می‌توان از
        # عملیات مورفولوژیک استفاده کرد. اما برای شروع همین کافی است.

        return m.unsqueeze(0).float() # خروجی (1, W, H, D) به صورت 0 و 1

    return Lambdad(keys=["brain_mask_tmp"], func=_f)
# section 5

def dilate_label_bin_lambdad(kernel: int = 7):
    assert kernel % 2 == 1, "kernel باید فرد باشد."
    pad = kernel // 2

    def _f(x):
        # x شکلش هست (1, W, H, D)
        if x.ndim == 3:
            x = x.unsqueeze(0)

        # اطمینان از باینری بودن و نوع داده
        x = (x > 0).float()

        # بردن به فرمت (N, C, D, H, W) که برای max_pool3d بهینه است
        # نکته: ترتیب W,H,D مهم نیست چون کرنل متقارن است، فقط باید 5 بعدی شود
        xb = x.unsqueeze(0) # (1, 1, W, H, D)

        # عملیات دیلیت (بزرگتر کردن ناحیه تومور)
        y = F.max_pool3d(xb, kernel_size=kernel, stride=1, padding=pad)

        # برگشت به شکل (1, W, H, D) و تبدیل به uint8 برای اشغال حافظه کمتر
        return y.squeeze(0).to(torch.uint8)

    return Lambdad(keys=["label_bin_tmp"], func=_f)
# =========================================================
# Section 6: Pre-Processing Transforms (TRAIN) — FIXED 💎
# =========================================================
from monai.transforms import (
    Compose, EnsureTyped, Orientationd, Spacingd, Lambdad,
    CropForegroundd, CopyItemsd, MaskIntensityd,
    NormalizeIntensityd, SpatialPadd
)

def build_pre_transform_train(cfg):
    return Compose([
        # 1. تبدیل به تنسور (با حفظ متادیتا! track_meta=True پیش‌فرض است)
        EnsureTyped(keys=["image", "label"], track_meta=True),

        # 2. جهت استاندارد
        Orientationd(keys=["image", "label"], axcodes="RAS"),

        # 3. ریسمپل (الان دیگه ارور نمیده چون لیبل هم کانال داره)
        Spacingd(
            keys=["image", "label"],
            pixdim=(1.0, 1.0, 1.0),
            mode=("bilinear", "nearest")
        ),

        # (دستور unsqueeze را حذف کردیم چون در دیتاست انجام دادیم)

        # 4. حذف حاشیه
        CropForegroundd(keys=["image", "label"], source_key="image"),

        # 5. نرمال‌سازی
        CopyItemsd(keys=["image"], names=["brain_mask_tmp"]),
        mk_brain_mask_lambdad(cfg),
        MaskIntensityd(keys=["image"], mask_key="brain_mask_tmp"),
        NormalizeIntensityd(keys=["image"], channel_wise=True, nonzero=True),
        MaskIntensityd(keys=["image"], mask_key="brain_mask_tmp"),

        # 6. پدینگ
        SpatialPadd(keys=["image", "label", "brain_mask_tmp"], spatial_size=cfg.patch_size),

        # 7. آماده‌سازی لیبل برای کراپ
        CopyItemsd(keys=["label"], names=["label_bin_tmp"]),
        Lambdad(keys=["label_bin_tmp"], func=lambda x: (x > 0).float()),
        dilate_label_bin_lambdad(kernel=getattr(cfg, "pos_dilate_kernel", 7)),
    ])
# =========================================================
# Section 7: Pre-Processing Transforms (VAL) — FIXED ✅
# =========================================================
from monai.transforms import (
    Compose, EnsureTyped, Orientationd, Spacingd, Lambdad,
    CropForegroundd, CopyItemsd, MaskIntensityd,
    NormalizeIntensityd, SpatialPadd, DeleteItemsd
)

def build_pre_transform_val(cfg):
    """
    پایپ‌لاین ولیدیشن (Validation):
    باید دقیقاً همان پیش‌پردازشی که روی Train انجام شده (نرمال‌سازی و ...)
    اینجا هم انجام شود، اما بدون آگمنتیشن‌های تصادفی (چرخش، نویز و ...).
    """
    return Compose([
        # 1. تبدیل به تنسور (حذف EnsureChannelFirst چون دیتاست خودش (4,...) می‌دهد)
        EnsureTyped(keys=["image", "label"], track_meta=True),

        # 2. جهت استاندارد (RAS)
        Orientationd(keys=["image", "label"], axcodes="RAS"),

        # 3. یکسان‌سازی رزولوشن
        Spacingd(
            keys=["image", "label"],
            pixdim=(1.0, 1.0, 1.0),
            mode=("bilinear", "nearest")
        ),

        # 4. اصلاح ابعاد لیبل: (W,H,D) -> (1,W,H,D)
        Lambdad(keys=["label"], func=lambda x: x.unsqueeze(0) if x.ndim == 3 else x),

        # 5. حذف حاشیه‌های خالی برای افزایش سرعت ولیدیشن
        CropForegroundd(keys=["image", "label"], source_key="image"),

        # 6. ساخت ماسک موقت مغز برای نرمال‌سازی
        CopyItemsd(keys=["image"], names=["brain_mask_tmp"]),
        mk_brain_mask_lambdad(cfg),

        # 7. نرمال‌سازی هوشمند (دقیقاً مشابه Train)
        MaskIntensityd(keys=["image"], mask_key="brain_mask_tmp"),
        NormalizeIntensityd(keys=["image"], channel_wise=True, nonzero=True),
        MaskIntensityd(keys=["image"], mask_key="brain_mask_tmp"),

        # 8. پدینگ ایمن (حیاتی)
        # اگر CropForegroundd تصویر را کوچکتر از سایز پچ کرد، پدینگ آن را درست می‌کند
        SpatialPadd(keys=["image", "label", "brain_mask_tmp"], spatial_size=cfg.patch_size),

        # 9. فرمت نهایی لیبل برای محاسبه Loss و Metric
        Lambdad(keys=["label"], func=lambda x: x.long()),
        # حذف بعد کانال لیبل چون Loss های MONAI معمولاً (B, W, H, D) می‌گیرند نه (B, 1, W, H, D)
        Lambdad(keys=["label"], func=lambda x: x.squeeze(0) if (x.ndim == 4 and x.shape[0] == 1) else x),

        # 10. پاکسازی حافظه
        DeleteItemsd(keys=["brain_mask_tmp"]),
    ])

# =========================================================
# Section 9: Heavy Augmentation Transform — SAFE ✅
# =========================================================
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    EnsureTyped,
    Orientationd,
    Spacingd,
    NormalizeIntensityd,
    CastToTyped,

    # ---------- Cropping / Sampling ----------
    RandCropByPosNegLabeld,
    RandSpatialCropd,

    # ---------- Intensity Augmentations ----------
    RandScaleIntensityd,
    RandShiftIntensityd,
    RandGaussianNoised,
    RandGaussianSmoothd,
    RandBiasFieldd,
    RandAdjustContrastd,   # ✅ این یکی برای ارور فعلی

    # ---------- Spatial Augmentations ----------
    RandFlipd,
    RandAffined,
    RandRotate90d,

    # ---------- Masking ----------
    MaskIntensityd,        # چون از brain_mask_tmp استفاده می‌کنی
)




def build_aug_transform(cfg):

    return Compose([
        RandCropByPosNegLabeld(
            keys=["image", "label", "brain_mask_tmp", "label_bin_tmp"],
            label_key="label_bin_tmp",
            spatial_size=cfg.patch_size,
            pos=2.0,
            neg=1.0,
            num_samples=getattr(cfg, "num_samples", 1),
            image_key="image",
            image_threshold=0,
        ),

        RandBiasFieldd(keys=["image"], prob=0.3, coeff_range=(0.2, 0.3)),

        RandFlipd(keys=["image", "label", "brain_mask_tmp"], prob=0.5, spatial_axis=0),
        RandFlipd(keys=["image", "label", "brain_mask_tmp"], prob=0.5, spatial_axis=1),
        RandFlipd(keys=["image", "label", "brain_mask_tmp"], prob=0.5, spatial_axis=2),
        RandRotate90d(keys=["image", "label", "brain_mask_tmp"], prob=0.5, max_k=3),

        RandGaussianNoised(keys=["image"], prob=0.15, mean=0.0, std=0.01),
        RandGaussianSmoothd(keys=["image"], prob=0.2, sigma_x=(0.5, 1.0), sigma_y=(0.5, 1.0), sigma_z=(0.5, 1.0)),

        RandScaleIntensityd(keys=["image"], factors=0.10, prob=0.5),
        RandShiftIntensityd(keys=["image"], offsets=0.10, prob=0.5),
        RandAdjustContrastd(keys=["image"], gamma=(0.5, 4.5), prob=0.5),

        MaskIntensityd(keys=["image"], mask_key="brain_mask_tmp"),

        Lambdad(keys=["label"], func=lambda x: x.long()),
        Lambdad(keys=["label"], func=lambda x: x.squeeze(0) if (x.ndim == 4 and x.shape[0] == 1) else x),

        DeleteItemsd(keys=["brain_mask_tmp", "label_bin_tmp"]),
        EnsureTyped(keys=["image", "label"]),
    ])
# =========================================================
# Section 10: Data Builders (Caching & Loading Strategy) ✅
# =========================================================
import random
import numpy as np
import torch
from pathlib import Path
from monai.data import (
    Dataset as MonaiDataset, CacheDataset, PersistentDataset,
    DataLoader, list_data_collate
)

# ---------------------------------------------------------
# Helper Functions for Reproducibility
# ---------------------------------------------------------
def _seed_worker(worker_id: int):
    """تنظیم سید برای هر ورکر جهت اطمینان از تکرارپذیری"""
    worker_seed = torch.initial_seed() % 2**32
    random.seed(worker_seed)
    np.random.seed(worker_seed)

def _make_generator(seed: int | None):
    """ساخت جنریتور رندوم برای دیتالودر"""
    if seed is None: return None
    g = torch.Generator()
    g.manual_seed(int(seed))
    return g

# ---------------------------------------------------------
# 1. Dataset Builder (Two-Stage Pipeline)
# ---------------------------------------------------------
def build_train_datasets(base_ds, cfg, pre_transform, aug_transform):
    """
    ساخت دیتاست آموزشی با استراتژی کشینگ (Cache Strategy).
    """
    backend = str(getattr(cfg, "cache_backend", "none")).lower()
    cache_rate = float(getattr(cfg, "cache_rate", 0.0))
    nw = int(getattr(cfg, "num_workers", 2))

    # --- مرحله 1: کش کردن دیتای اولیه ---
    # اگر کش ریت 0 باشد، حتی اگر رم انتخاب شده باشد، باید کش غیرفعال شود
    if backend == "ram" and cache_rate > 0:
        print(f"🚀 [Cache] Loading {cache_rate*100}% data into RAM with {nw} workers...")
        pre_ds = CacheDataset(
            data=base_ds,
            transform=pre_transform,
            cache_rate=cache_rate,
            num_workers=nw,
        )
    elif backend == "disk":
        cache_dir = Path(getattr(cfg, "cache_dir", "./cache_pre"))
        cache_dir.mkdir(parents=True, exist_ok=True)
        print(f"💾 [Cache] Using Disk Cache at: {cache_dir}")
        pre_ds = PersistentDataset(
            data=base_ds,
            transform=pre_transform,
            cache_dir=str(cache_dir),
        )
    else:
        # حالت None یا وقتی که cache_rate=0 است
        if backend == "ram": print("⚠️ Cache Rate is 0.0 -> Switching to standard Dataset.")
        pre_ds = MonaiDataset(data=base_ds, transform=pre_transform)

    # --- مرحله 2: اعمال آگمنتیشن‌های تصادفی روی دیتای کش شده ---
    train_ds = MonaiDataset(data=pre_ds, transform=aug_transform)
    return train_ds

# ---------------------------------------------------------
# 2. Train Loader Builder
# ---------------------------------------------------------
def build_train_loader(train_ds, cfg):
    """
    ساخت دیتالودر مخصوص آموزش با list_data_collate
    """
    bs = int(getattr(cfg, "batch_size", 1))
    nw = int(getattr(cfg, "num_workers", 2))
    pin = bool(getattr(cfg, "pin_memory", True))
    seed = getattr(cfg, "seed", None)

    # اگر تعداد ورکرها 0 باشد، persistent_workers باید خاموش باشد
    persistent = (nw > 0)

    return DataLoader(
        dataset=train_ds,
        batch_size=bs,
        shuffle=True,
        num_workers=nw,
        pin_memory=pin,
        drop_last=False,
        # حیاتی: چون خروجی RandCrop یک لیست است، باید با این تابع جمع شود
        collate_fn=list_data_collate,
        persistent_workers=persistent,
        worker_init_fn=_seed_worker,
        generator=_make_generator(seed),
        prefetch_factor=2 if nw > 0 else None
    )

# ---------------------------------------------------------
# 3. Validation Loader Builder
# ---------------------------------------------------------
def build_val_loader(val_base, cfg, pre_val_transform):
    """
    ساخت دیتالودر ولیدیشن (بدون شافل، بچ سایز 1)
    """
    # ولیدیشن آگمنتیشن ندارد، فقط pre_transform
    val_ds = MonaiDataset(data=val_base, transform=pre_val_transform)

    nw = int(getattr(cfg, "num_workers", 2))
    pin = bool(getattr(cfg, "pin_memory", True))
    seed = getattr(cfg, "seed", None)
    persistent = (nw > 0)

    return DataLoader(
        dataset=val_ds,
        batch_size=1, # ولیدیشن همیشه تک‌تصویر است
        shuffle=False,
        num_workers=nw,
        pin_memory=pin,
        persistent_workers=persistent,
        worker_init_fn=_seed_worker,
        generator=_make_generator(seed),
        prefetch_factor=2 if nw > 0 else None
    )

# =========================================================
# Section 11: Loader Builder — CLEAN VERSION ✅
# =========================================================

def build_loaders(cfg, data_root=None):

    if data_root is None:
        data_root = cfg.dataset_path

    print(f"📂 Loading Dataset from: {data_root}")

    # 1. ساخت دیتاست‌های پایه (Passing explicit configs)
    # حتماً kfold و seed را از cfg بگیرید تا تقسیم‌بندی درست باشد
    train_base = BraTSDictDataset(
        dataset_path=data_root,
        training=True,
        kfold=getattr(cfg, "kfold", 5),
        fold=getattr(cfg, "fold", 0),
        seed=getattr(cfg, "seed", 42),
        verbose=True
    )

    val_base = BraTSDictDataset(
        dataset_path=data_root,
        training=False,
        kfold=getattr(cfg, "kfold", 5),
        fold=getattr(cfg, "fold", 0),
        seed=getattr(cfg, "seed", 42),
        verbose=True
    )

    print(f"📊 [Stats] Train: {len(train_base)} | Val: {len(val_base)}")

    # 2. ساخت ترنسفرم‌ها
    print("⚙️ Building Transforms...")
    pre_tr  = build_pre_transform_train(cfg)
    aug_tr  = build_aug_transform(cfg)
    pre_val = build_pre_transform_val(cfg)

    # 3. ترکیب و ساخت لودرها
    print("🚀 Initializing Pipeline...")

    # ساخت دیتاست نهایی (شامل کشینگ در صورت نیاز)
    train_ds = build_train_datasets(train_base, cfg, pre_tr, aug_tr)

    # ساخت دیتالودرها
    train_loader = build_train_loader(train_ds, cfg)
    val_loader = build_val_loader(val_base, cfg, pre_val)

    print("✅ All Loaders Ready!")
    return train_loader, val_loader


cfg = CFG()


#ZIP_PATH   = "/content/data/BraTS/BraTS2021_Training_Data.tar"

ZIP_PATH   = "/content/drive/My Drive/brats2021_20.zip"

EXTRACT_TO = "/content/data/BraTS"      # این‌جا بدون تودرتو می‌ریزه
train_cache  = "/content/cache_pre_train"
val_cache    = "/content/cache_pre_val"

dataset_root = prepare_dataset_from_archive(
    cfg,
    archive_path=ZIP_PATH,
    extract_dir=EXTRACT_TO,
    ensure_cache_dirs=(train_cache, val_cache),
    force_reextract=True,
    )



cfg.dataset_path = dataset_root
print(cfg.dataset_path)

'''